# 한국로텍 데이터 분석

검사 결과와 생산·도금 기록을 모아 불량 현황과 작업조건의 관계를 살펴봅니다.

## 접속과 자료 보관

회사 자료는 GitHub에 올리지 않고 서버에 따로 보관합니다. 연구실 서버의 Docker에 Jupyter와 실제 자료가 준비되어 있어 설치나 SSH 접속 없이 이용할 수 있습니다.
담당자에게 받은 접속용 링크 전체를 엽니다. 접속 정보가 포함된 링크는 별도 로그인 코드 입력 없이 열립니다. 처음 접속할 때 안내 화면이 나오면 **Visit Site**를 누릅니다.

## 결과 보기

공유받은 서버 링크에서는 설치나 파일 준비 없이 이용할 수 있습니다.
아래로 내려가면 항목별 결과가 나옵니다. 전체 내용을 한 번에 보려면 [분석 보고서 열기](분석결과/한국로텍_분석보고서.html)를 누르세요.

## 분석 다시 실행하기

1. 화면 맨 위의 **Run(실행)** 메뉴를 누릅니다.
2. **Run All Cells(모든 셀 실행)**를 누릅니다.
3. 맨 아래에 **분석 완료**가 나올 때까지 기다립니다. 서버에서는 약 3~5분 걸립니다.

코드를 입력하거나 수정할 필요는 없습니다. 코드는 접어 두었으며, 접힌 상태에서도 전체 실행됩니다.
다시 실행하면 `분석결과` 폴더의 결과가 갱신됩니다.

## 새 자료 추가하기

왼쪽 `추가자료/양식`에서 CSV 또는 Excel 양식을 내려받아 작성하고, `추가자료/입력` 폴더에 올린 뒤 전체 실행합니다. 검사 결과가 있는 자료는 기존 자료와 함께 모델을 다시 학습하고, 검사 결과가 없는 자료는 예측만 계산합니다.
양식의 열 이름과 단위를 그대로 사용합니다. 기존 회사 원본 Excel은 담당자가 양식에 맞게 정리한 뒤 넣습니다. [새 자료 추가 안내](https://github.com/realironn/Hkrotec-Data-Analysis/blob/main/docs/additional-data.md)

## 실행이 멈췄을 때

빨간 오류 메시지가 나오면 해당 화면을 담당자에게 전달해 주세요.
`Kernel(커널)` 선택 창이 나오면 **Python 3 (ipykernel)**을 선택합니다.

내 PC에서 직접 실행하려면 [Windows 설치 안내](https://github.com/realironn/Hkrotec-Data-Analysis/blob/main/docs/windows-setup.md)를 참고해 주세요. PC 실행에는 별도로 받은 회사 자료와 Python 3.14가 필요합니다.


In [ ]:
# 분석에 필요한 프로그램을 준비합니다.
import sys, json, subprocess, os
from pathlib import Path

if sys.version_info[:2] != (3, 14):
    raise RuntimeError('Python 3.14가 필요합니다. 서버 이용자는 이 오류 화면을 담당자에게 전달해 주세요. PC 이용자는 Windows 설치 안내를 확인해 주세요.')

_nb_name = '한국로텍_데이터분석.ipynb'
_nb_candidates = [p/q/_nb_name for p in [Path.cwd(), *Path.cwd().parents]
                  for q in [Path('.'), Path('04_노트북_실행용'), Path('한국로텍_최종/04_노트북_실행용')]]
_nb_file = next((p for p in _nb_candidates if p.is_file()), None)
if _nb_file is None:
    raise FileNotFoundError('노트북 파일을 찾을 수 없습니다. 이 오류 화면을 담당자에게 전달해 주세요.')
_runtime = json.loads(_nb_file.read_text(encoding='utf-8'))['metadata']['rotec_runtime']
_install_packages = _runtime['requirements_lock']
if not _install_packages or not all(isinstance(p,str) and '==' in p for p in _install_packages):
    raise ValueError('프로그램 설치 정보가 올바르지 않습니다. 노트북 파일을 담당자에게 다시 받아 주세요.')
if os.environ.get('ROTEC_PREINSTALLED') == '1':
    # 서버에서는 미리 설치된 버전을 확인합니다.
    from importlib.metadata import version
    from packaging.requirements import Requirement
    for entry in _install_packages:
        requirement = Requirement(entry)
        if requirement.marker is None or requirement.marker.evaluate():
            if version(requirement.name) not in requirement.specifier:
                raise RuntimeError('서버의 프로그램 버전을 확인해야 합니다. 담당자에게 전달해 주세요: ' + requirement.name)
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           '--disable-pip-version-check', '--quiet', *_install_packages])
print('실행 준비가 끝났습니다.')


## 1. 자료 확인

분석에 필요한 파일이 있는지, 원본과 일치하는지 확인합니다. 서버에는 담당자가 자료를 준비해 두었습니다.


In [ ]:
"""원본과 기존 검토자료를 대조합니다."""
from pathlib import Path
import base64,csv,hashlib,io,json,zipfile

def _file_hash(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()

def _prepare_original_inputs(root,out,runtime):
    root=Path(root).resolve();out=Path(out).resolve()
    if not out.is_relative_to(root):raise ValueError('결과는 입력 폴더 안에 저장해 주세요.')
    out.mkdir(parents=True,exist_ok=True)
    reference_path=root/runtime['private_reference_file']
    if not reference_path.is_file():
        raise FileNotFoundError('분석참고자료 파일이 없습니다. 별도로 받은 한국로텍_분석참고자료.json을 prepared 폴더 안에 넣어 주세요.')
    reference_bytes=reference_path.read_bytes()
    if hashlib.sha256(reference_bytes).hexdigest()!=runtime['private_reference_sha256']:
        raise ValueError('분석참고자료의 버전이 노트북과 다릅니다. 담당자가 제공한 같은 배포본의 파일인지 확인해 주세요.')
    reference=json.loads(reference_bytes)
    bundle=reference['embedded_review']
    payload=base64.b64decode(bundle['zip_base64'],validate=True)
    if hashlib.sha256(payload).hexdigest()!=bundle['zip_sha256']:
        raise ValueError('분석참고자료에 손상이 있습니다. 담당자에게 파일 확인을 요청해 주세요.')
    written=[]
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            if member.is_dir():continue
            relative=Path(member.filename)
            if relative.is_absolute() or '..' in relative.parts:
                raise ValueError('분석참고자료의 파일 위치가 올바르지 않습니다. 담당자에게 확인을 요청해 주세요.')
            base=out
            target=(base/relative).resolve()
            if not target.is_relative_to(base):raise ValueError('실행자료 경로를 확인해 주세요.')
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(archive.read(member))
            written.append(target)
    sources={}
    embedded_sources=bundle['embedded_source_paths']
    reference_usage=[]
    failures=[]
    for item in bundle['expected_inputs']:
        rel=item['file'];actual=(root/rel).resolve()
        if not actual.is_relative_to(root):raise ValueError('입력 경로가 분석 폴더 밖입니다.')
        mode='받은 입력자료'
        if not actual.is_file() and rel in embedded_sources:
            actual=(out/embedded_sources[rel]).resolve()
            if not actual.is_relative_to(out):raise ValueError('내장 기준자료 경로 오류')
            mode='노트북에 포함한 기존 과제·테마 기준'
        if not actual.is_file():failures.append(rel+' : 파일 없음');continue
        if _file_hash(actual)!=item['sha256']:
            failures.append(rel+' : 검토 기준과 내용 다름');continue
        sources[rel]=actual
        if rel in embedded_sources:
            reference_usage.append({'기준자료':rel,'사용위치':actual.relative_to(root).as_posix(),'구분':mode,'SHA256':item['sha256']})
    if failures:
        raise ValueError('아래 입력파일이 없거나 기존 자료와 다릅니다. 파일 목록을 담당자에게 전달해 주세요.\n'+'\n'.join(failures[:12]))
    expected_assets=bundle['expected_image_assets']
    image_dir=out/'이미지검토'
    assets_dir=image_dir/'01_추출이미지'
    # 원본과 일치하는 이미지가 있으면 다시 추출하지 않습니다.
    cached=all((assets_dir/item['filename']).is_file() and _file_hash(assets_dir/item['filename'])==item['sha256'] for item in expected_assets)
    if not cached:
        _extract_original_images(root,image_dir,bundle['review_inventory'],sources)
    actual_assets={p.name:_file_hash(p) for p in assets_dir.iterdir() if p.is_file()}
    if any(actual_assets.get(item['filename'])!=item['sha256'] for item in expected_assets):
        raise ValueError('원본에서 다시 추출한 이미지가 기존 검토 기록과 다릅니다. 이미지 재검토가 필요합니다.')
    from PIL import Image
    rendered=image_dir/'02_열람용이미지';rendered.mkdir(exist_ok=True)
    fallback=set(bundle['embedded_image_renders'])
    for item in expected_assets:
        original=assets_dir/item['filename'];target=rendered/(original.stem+'.png')
        if original.name in fallback:
            if not target.is_file():raise ValueError('검증된 열람 이미지가 없습니다: '+original.name)
            continue
        with Image.open(original) as picture:
            picture.convert('RGBA').save(target,format='PNG')
    source_note={
        '검사방식':'원본 파일 해시, 추출 이미지, 숫자 셀 대조',
        '자동_OCR_실행':False,
        '원본이미지_해시대조수':len(expected_assets),
        '검토기준자료_원본수':len(bundle['review_inventory']),
        '기준문서_사용내역':reference_usage,
    }
    (image_dir/'현재실행_근거설명.json').write_text(json.dumps(source_note,ensure_ascii=False,indent=2),encoding='utf-8')
    pd.DataFrame(reference_usage).to_csv(out/'tables/실행기준_사용내역.csv',index=False,encoding='utf-8-sig')
    return {'sources':sources,'image_audit':image_dir,'parameters':reference['analysis_parameters'],
            'review_inventory':bundle['review_inventory'],'source_note':source_note}



def _extract_original_images(root,output,review_inventory,sources):
    """원본 문서에 들어 있는 이미지와 위치를 확인합니다."""
    from pathlib import Path
    from collections import Counter, defaultdict
    import csv, hashlib, io, json, posixpath, struct, zipfile, zlib, re, os
    import xml.etree.ElementTree as ET
    from PIL import Image, ImageOps, ImageDraw, ImageFont
    from xlrd.compdoc import CompDoc

    ROOT=Path(root)
    OUT=Path(output)
    ASSETS = OUT / '01_추출이미지'
    WORK=OUT/'05_재실행도구'
    for p in [OUT,ASSETS,WORK]:p.mkdir(parents=True,exist_ok=True)
    NS = {'r':'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
          'a':'http://schemas.openxmlformats.org/drawingml/2006/main',
          'xdr':'http://schemas.openxmlformats.org/drawingml/2006/spreadsheetDrawing',
          's':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    inventory=review_inventory
    assets, occurrences, file_audit, external, embeddings = {}, [], [], [], []

    def rel_source(rel_path):
        if rel_path == '_rels/.rels': return ''
        parent, leaf = posixpath.split(rel_path)
        return posixpath.join(posixpath.dirname(parent), leaf[:-5])

    def sha(data): return hashlib.sha256(data).hexdigest()

    def image_type(data, fallback=''):
        if data.startswith(b'\x89PNG\r\n\x1a\n'): return '.png'
        if data.startswith(b'\xff\xd8\xff'): return '.jpg'
        if data.startswith((b'GIF87a',b'GIF89a')): return '.gif'
        if data.startswith(b'BM'): return '.bmp'
        if data.startswith((b'II*\x00',b'MM\x00*')): return '.tif'
        return fallback or '.bin'

    def register(data, suffix, source, media, locations):
        h = sha(data)
        if h not in assets:
            uid = f'IMG-{len(assets)+1:03d}'
            suffix = image_type(data,suffix.lower())
            dst = ASSETS / (uid+suffix)
            dst.write_bytes(data)
            size, mode, error = None, None, ''
            try:
                with Image.open(io.BytesIO(data)) as im: size,mode=im.size,im.mode
            except Exception as e: error=str(e)
            assets[h] = {'id':uid,'sha256':h,'path':str(dst.relative_to(OUT)),'bytes':len(data),'width':size[0] if size else None,'height':size[1] if size else None,'mode':mode,'read_error':error}
        for loc in locations or ['참조 위치 미확인']:
            occurrences.append({'image_id':assets[h]['id'],'source_file':source,'container_media':media,'location':loc,'sha256':h})

    def biff_pictures(data, source):
        stream = CompDoc(data).get_named_stream('Workbook')
        if stream is None: stream = CompDoc(data).get_named_stream('Book')
        if stream is None: raise ValueError('Workbook/Book stream absent')
        counts, blocks, pos, current, hf = Counter(), [], 0, None, bytearray()
        while pos+4<=len(stream):
            kind,length=struct.unpack_from('<HH',stream,pos); pos+=4
            body=stream[pos:pos+length];pos+=length
            counts[f'0x{kind:04X}']+=1
            if kind==0x00EB:
                if current is None:current=bytearray();blocks.append(current)
                current.extend(body)
            elif kind==0x003C and current is not None:
                current.extend(body)
            else: current=None
            if kind==0x0866:hf.extend(body)
        (WORK/(Path(source).stem+'_biff_records.json')).write_text(json.dumps(dict(counts),indent=2),encoding='utf-8')
        if hf:blocks.append(hf)
        blip_count=0
        declared_blips=0
        for block_no,block in enumerate(blocks):
            for store in re.finditer(b'\x01\xf0',block):
                q=store.start()-2
                if q>=0 and q+8<=len(block):
                    vi,kt,ln=struct.unpack_from('<HHI',block,q)
                    if (vi&15)==15 and ln<=len(block)-q-8:declared_blips+=vi>>4
            for hit in re.finditer(b'[\x1a-\x1f\x29]\xf0',block):
                q=hit.start()-2
                if q<0 or q+8>len(block):continue
                verinst,kind,length=struct.unpack_from('<HHI',block,q)
                if length>len(block)-q-8:continue
                body=bytes(block[q+8:q+8+length])
                if kind in (0xF01D,0xF01E,0xF01F,0xF029):
                    for sig,suffix in [(b'\x89PNG\r\n\x1a\n','.png'),(b'\xff\xd8\xff','.jpg')]:
                        off=body.find(sig,0,80)
                        if off>=0:
                            register(body[off:],suffix,source,f'BIFF/OfficeArt/block{block_no}/0x{kind:04X}',['BIFF 도형/머리글; 시트 위치 미확인'])
                            blip_count+=1
                            break
                    else: embeddings.append({'source_file':source,'part':f'OfficeArt 0x{kind:04X}','bytes':length,'issue':'Raster BLIP without expected signature'})
                elif kind in (0xF01A,0xF01B,0xF01C):
                    decoded=False
                    for uidbytes in [16,32]:
                        if len(body)<uidbytes+34:continue
                        cbsize=struct.unpack_from('<I',body,uidbytes)[0]
                        cbsave=struct.unpack_from('<I',body,uidbytes+28)[0]
                        compression,filterbyte=body[uidbytes+32:uidbytes+34]
                        payload=body[uidbytes+34:uidbytes+34+cbsave]
                        if filterbyte!=0xFE or compression not in (0,0xFE) or len(payload)!=cbsave:continue
                        try:picture=zlib.decompress(payload) if compression==0 else payload
                        except zlib.error:continue
                        if len(picture)!=cbsize:continue
                        register(picture,{0xF01A:'.emf',0xF01B:'.wmf',0xF01C:'.pict'}[kind],source,f'BIFF/OfficeArt/block{block_no}/0x{kind:04X}',['BIFF 도형/머리글; 시트 위치 미확인'])
                        blip_count+=1;decoded=True;break
                    if not decoded:embeddings.append({'source_file':source,'part':f'OfficeArt 0x{kind:04X}','bytes':length,'issue':'Metafile BLIP; further decode required'})
        return {'biff_records':sum(counts.values()),'drawing_groups':counts['0x00EB'],'drawing_records':counts['0x00EC'],'legacy_image_records':counts['0x007F'],'header_image_records':counts['0x0866'],'declared_blips':declared_blips,'extracted_blips':blip_count}

    for inv in inventory:
        rel=inv['source_file'];p=sources[rel];data=p.read_bytes()
        assert sha(data)==inv['sha256'],f'original changed: {rel}'
        before=len(occurrences)
        row={'source_file':rel,'extension':p.suffix.lower(),'media_parts':0,'sha256_matches':True,'errors':''}
        if zipfile.is_zipfile(io.BytesIO(data)):
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                names=set(z.namelist());rels={};reverse=defaultdict(set);roots={};anchors=defaultdict(list)
                for name in sorted(names):
                    if not name.endswith('.rels'):continue
                    parent=rel_source(name);mapping={}
                    for node in ET.fromstring(z.read(name)):
                        if node.attrib.get('TargetMode')=='External':
                            if '/image' in node.attrib.get('Type',''):external.append({'source_file':rel,'part':parent,**node.attrib})
                            continue
                        target=posixpath.normpath(posixpath.join(posixpath.dirname(parent),node.attrib['Target'].lstrip('/'))) if not node.attrib['Target'].startswith('/') else node.attrib['Target'].lstrip('/')
                        mapping[node.attrib['Id']]=target;reverse[target].add(parent)
                    rels[parent]=mapping
                if 'xl/workbook.xml' in names:
                    for s in ET.fromstring(z.read('xl/workbook.xml')).findall('s:sheets/s:sheet',NS):
                        target=rels.get('xl/workbook.xml',{}).get(s.attrib['{'+NS['r']+'}id'])
                        if target:roots[target]='시트: '+s.attrib['name']
                for name in names:
                    if name.startswith('ppt/slides/slide') and name.endswith('.xml') and '/_rels/' not in name:
                        roots[name]='슬라이드 '+Path(name).stem.replace('slide','')
                    if name.startswith('xl/drawings/drawing') and name.endswith('.xml'):
                        for anchor in ET.fromstring(z.read(name)):
                            origin=anchor.find('xdr:from',NS)
                            loc=''
                            if origin is not None:
                                col=int(origin.find('xdr:col',NS).text);rr=int(origin.find('xdr:row',NS).text)+1
                                cc='';col+=1
                                while col:col,rem=divmod(col-1,26);cc=chr(65+rem)+cc
                                loc=cc+str(rr)
                            for b in anchor.findall('.//a:blip',NS):
                                target=rels.get(name,{}).get(b.attrib.get('{'+NS['r']+'}embed',''))
                                if target:anchors[target].append((name,loc))
                def trace_roots(target):
                    found=set();seen=set();queue=[target]
                    while queue:
                        node=queue.pop()
                        if node in seen:continue
                        seen.add(node)
                        if node in roots:found.add(roots[node])
                        queue.extend(reverse.get(node,()))
                    return found
                def locations(target):
                    if anchors.get(target):
                        return sorted({loc+' / '+a for drawing,a in anchors[target] for loc in trace_roots(drawing)})
                    found=trace_roots(target)
                    return sorted(found) or (['문서 축소판'] if target.startswith('docProps/') else ['참조 위치 미확인'])
                media=[n for n in sorted(names) if '/media/' in n or n.startswith('docProps/thumbnail')]
                row['media_parts']=len(media)
                for name in media:register(z.read(name),Path(name).suffix,rel,name,locations(name))
                for name in sorted(names):
                    if '/embeddings/' in name:
                        nested=z.read(name);nested_media=[]
                        if zipfile.is_zipfile(io.BytesIO(nested)):
                            with zipfile.ZipFile(io.BytesIO(nested)) as nz:
                                for nn in nz.namelist():
                                    if '/media/' in nn:
                                        nested_media.append(nn);register(nz.read(nn),Path(nn).suffix,rel,name+'/'+nn,['내장 문서 / '+name])
                        embeddings.append({'source_file':rel,'part':name,'bytes':len(nested),'issue':'Embedded Office document; checked','image_parts':nested_media})
        elif p.suffix.lower()=='.xls':
            row.update(biff_pictures(data,rel))
        else: row['errors']='Unrecognized file container'
        row['image_occurrences']=len(occurrences)-before
        file_audit.append(row)

    result={'files':file_audit,'assets':list(assets.values()),'occurrences':occurrences,'external_images':external,'embedded_objects':embeddings}
    (WORK/'extraction.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
    print(f'원본 문서 {len(file_audit)}개에서 이미지 {len(assets)}개를 확인했습니다.')

    return result


In [ ]:
from pathlib import Path
from datetime import datetime
import csv, re, json, hashlib, sys, platform, base64, html, warnings, zipfile, os, shutil
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Markdown, Image
from openpyxl import load_workbook
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

_required_inputs=['raw','csv/manifest.csv','prepared/five_fold/daily_targets.csv','prepared/five_fold/fold_assignments.csv']
_root_candidates=[q for p in [_nb_file.parent,Path.cwd(),*Path.cwd().parents]
                  for q in [p,p/'04_노트북_실행용',p/'한국로텍_최종/04_노트북_실행용']]
ROOT=next((p.resolve() for p in _root_candidates if all((p/r).exists() for r in _required_inputs)),None)
if ROOT is None:
    raise FileNotFoundError('raw·csv·prepared 폴더와 노트북을 같은 위치에 두세요.')
_default_output='분석결과_재실행' if (ROOT/'03_검증도구').is_dir() else '분석결과'
OUT=Path(os.environ.get('ROTEC_RESULTS_DIR',ROOT/_default_output)).resolve()
if not OUT.is_relative_to(ROOT):raise ValueError('출력은 현재 입력 폴더 안에 지정하세요.')
TABLES,FIGURES=OUT/'tables',OUT/'figures'
for p in [OUT,TABLES,FIGURES]:p.mkdir(parents=True,exist_ok=True)
_PREPARED_INPUTS=_prepare_original_inputs(ROOT,OUT,_runtime)
IMAGE_AUDIT=_PREPARED_INPUTS['image_audit']
def _input_file(relative):
    return _PREPARED_INPUTS['sources'].get(str(relative),ROOT/relative)

ITEMS = list(_PREPARED_INPUTS['parameters']['items'])
LAGS = [0,5,10]  # 검사일과 같은 날, 5일 전, 10일 전의 작업일을 비교합니다.
VOLTAGE_FLOOR = 0.1  # 추가 비교에서는 0.1V 이하를 제외합니다. 원래 값은 보관합니다.
SEED = 20261012
np.random.seed(SEED)
font_path = Path('C:/Windows/Fonts/malgun.ttf')
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    plt.rcParams['font.family']='Malgun Gothic'
else:
    available_fonts={font.name for font in font_manager.fontManager.ttflist}
    selected_font=next((name for name in ['Noto Sans CJK KR','Noto Sans CJK JP','NanumGothic'] if name in available_fonts),None)
    if selected_font:plt.rcParams['font.family']=selected_font
plt.rcParams.update({'axes.unicode_minus':False,'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False})
pd.set_option('display.max_columns',18)

def save(df,name):
    df.to_csv(TABLES/(name+'.csv'),index=False,encoding='utf-8-sig',date_format='%Y-%m-%d')
    return df
def rows_for(path):
    with Path(path).open(encoding='utf-8-sig',newline='') as f: return list(csv.reader(f))
def num(v):
    try: return float(str(v).replace(',','').replace('₩','').strip())
    except (ValueError,TypeError): return np.nan
def isdate(v): return bool(re.match(r'^20\d{2}-\d{2}-\d{2}',str(v)))
def parts(v): return [x.strip() for x in str(v).replace('_x000D_','').splitlines() if x.strip()]
def slot(v,k):
    p=parts(v); return num(p[k]) if len(p)>k else np.nan
def sec(v):
    s=str(v).strip(); m=re.fullmatch(r'(-?)(\d+):(\d{2}):(\d{2})',s)
    return (-1 if m[1] else 1)*(int(m[2])*3600+int(m[3])*60+int(m[4])) if m else np.nan
def digest(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
manifest=pd.read_csv(ROOT/'csv/manifest.csv',keep_default_na=False)
print('입력자료 확인이 끝났습니다.')
# 화면에 표시하는 표 제목을 한글로 바꿉니다.
TABLE_LABELS = {'month': '월',
 'calendar_days': '날짜 수',
 'valid_days': '유효 일수',
 'defects': '불량건수',
 'reviewed': '검사수량',
 'pinhole_pit': '핀홀·피트 건수',
 'thickness_under': '두께미달 건수',
 'total_pphv': '불량기록 비율(%)',
 'pinhole_pit_pphv': '핀홀·피트 기록 비율(%)',
 'join': '연결 정의',
 'denominator_name': '전체 기준',
 'denominator': '전체 기록 수',
 'matched': '연결된 기록 수',
 'rate': '연결 비율',
 'reason': '한계',
 'item': '품번',
 'events': '이력 건수',
 'fb_ids': '설비 운행ID 수',
 'first_date': '시작일',
 'last_date': '종료일',
 'days': '공정 존재일',
 'logged_parts': '기록 수량 합계',
 'low_voltage_events': '저전압 기록',
 'station': '스테이션',
 'low_voltage': '저전압 기록',
 'voltage_raw_median': '원전압 중앙값(V)',
 'voltage_clean_median': '진단 제외 후 전압(V)',
 'current_median': '전류 중앙값(A)',
 'temperature_median': '온도 중앙값(℃)',
 'low_voltage_fraction': '저전압 비율',
 'feature': '변수',
 'lag_days': '시차(일)',
 'n_days': '분석 일수',
 'spearman_r': '상관계수 r',
 'month_weekday_volume_adjusted_r': '월·요일·물량 통제 r',
 'q_bh': 'BH 보정 q',
 'line': '라인',
 'bath': '액조',
 'component': '성분',
 'measured': '실측 수',
 'spec_evaluable': '규격 판정 수',
 'spec_exceed': '규격 이탈 수',
 'management_evaluable': '관리치 판정 수',
 'management_exceed': '관리치 이탈 수',
 'measurements': '실측 수',
 'same_day_inspection_days': '검사일과 겹치는 날',
 'chemical': '약품 구분',
 'daily_rows': '일계 수',
 'monthly_minus_daily': '월계-일계(ml)',
 'daily_minus_events': '일계-투입내역(ml)',
 'model': '모형',
 'validation_days': '검증 일수',
 'fold_count': '검증 월 수',
 'mae_pp': '평균 예측 오차(%p)',
 'rmse_pp': '평균 제곱근오차(%p)',
 'weighted_mae_pp': '검사량 가중오차(%p)',
 'r2': '결정계수 R²',
 'train_days': '학습 일수',
 'test_days': '시험 일수',
 'defects_ppt': 'PPT 공정불량 수',
 'production_ppt': 'PPT 생산수량',
 'ppm_reported': 'PPT 실적(ppm)',
 'check': '검증 항목',
 'passed': '통과',
 'detail': '내용',
 'title': '원본 제목',
 'date_columns': '해당월 날짜 열',
 'excluded_other_month_dates': '제외한 다른달 날짜',
 'sheet_total': '원시트 불량 월계',
 'date_column_total': '해당월 불량 재집계',
 'sheet_reviewed': '원시트 검사 월계',
 'date_column_reviewed': '해당월 검사 재집계',
 'formula_error_cells': '수식 오류 셀',
 'missing_reviewed_dates': '검사수량 결측 일수',
 'net_ml_per_kwh__etch_chromic': '에칭 크롬산 순투입(ml/kWh)',
 'net_ml_per_kwh__chrome_chromic': '도금 크롬산 순투입(ml/kWh)',
 'automatic_ml_per_kwh__etch_chromic': '에칭 크롬산 자동투입(ml/kWh)',
 'automatic_ml_per_kwh__chrome_chromic': '도금 크롬산 자동투입(ml/kWh)',
 'additive_net_ml': '첨가제 순투입(ml)',
 'additive_automatic_ml': '첨가제 자동투입(ml)',
 'source_group': '자료 종류',
 'files': '파일 수',
 'tables': '계산표 수',
 'converted_rows': '변환한 기록 수',
 'daily_defects': '일별 검사 결과',
 'production_history': '생산 작업 기록',
 'plating_solution': '도금액 성분',
 'chemical_inputs': '약품 투입 기록',
 'power_usage': '전력 사용 기록',
 'date': '날짜',
 'total_defects': '불량기록 건수',
 'total_reviewed': '검사수량',
 'pinhole_pit_count': '핀홀·피트 건수',
 'thickness_under_count': '두께미달 건수',
 'pphv': '불량기록 비율(%)',
 'pphv_source': '원본의 불량기록 비율(%)',
 'density_set_median': '설정 전류밀도 중앙값',
 'density_actual_median': '실제 전류밀도 중앙값',
 'duration_median': '처리시간 중앙값(초)',
 'current_set_median': '설정전류 중앙값(A)',
 'current_actual_median': '실제전류 중앙값(A)',
 'voltage_median': '전압 중앙값(V)',
 'part_number': '품번',
 'source_sheet': '원본 시트',
 'source_cell': '원본 칸',
 'value': '측정값',
 'unit': '단위',
 'spec_source': '원본 규격',
 'before': '처리 전',
 'after': '처리 후',
 'computed_efficiency_percent': '계산한 정화율(%)',
 'threshold_comparison': '기준과 비교',
 'sample': '비교 날짜 기준',
 'target': '비교 대상',
 'chrome_events': '크롬 공정 기록 수',
 'sum': '합계',
 'max': '최댓값',
 'spearman_p': '상관관계 검정 p값',
 'abs_r': '상관계수의 크기',
 'item_group': '품목 구분',
 'stage': '분석 구간',
 'lag': '작업일 차이(일)'}

def show(*tables):
    for table in tables:
        display(table.rename(columns=TABLE_LABELS, index=TABLE_LABELS))


## 2. 분석 기준

분석할 품목과 불량 항목, 계산 기준을 확인합니다.
작은 구멍 형태의 불량인 **핀홀·피트**는 원본 검사표에서 한 항목으로 기록되어 함께 집계합니다.
**전류밀도**는 제품 표면적당 흐르는 전류의 크기입니다.


In [ ]:
requirements=pd.DataFrame([
 ['대표품목','; '.join(ITEMS),'기업 확정','테마-1 A9:A10 (RR는 2개 실제 품번)'],
 ['대상 불량','도금-핀홀/피트','기업 확정','테마-1 A17; 검사자료 코드21'],
 ['두께','다른 6품번 두께 기록 21개·시트 SPEC 존재. 대표 3품번 실측 0개, 단위 미표기','표본·정의 제한','2026년 액관리 이력대장 마이크로크랙 3시트 L열; 검사 코드26'],
 ['전류밀도','설정전류(A)/기하학적 표면(dm²); 실제평균도 보조 계산','기업 수식','테마-1 A13 및 삽입 이미지'],
 ['시차','검사일-공정일 = 0,5,10 달력일','실행과제','한국로텍 실행과제.txt'],
 ['불량 지표','불량건수/검사수량×100 (PPHV)','원자료 계산 재현','중복불량 여부 미확정. 고유 불량품률로 단정하지 않음'],
 ['공정-액조 매핑','스테이션 14/15/16과 액관리 1/3/4라인의 대응 없음','미확정','확정 후에만 개별 도금액 노출량 연결 가능'],
 ['합격·경보 기준','대표품목 판정기준·관리 조치 임계값 미확정. 기타품목의 시트 SPEC 존재','미확정','상관관계로 생산조건 또는 합격기준을 새로 확정하지 않음'],
 ['기존 분석','prepared/five_fold의 표본 및 집계 재현 가능; 기존 모델·계수·결과 보고서 없음','부분 재현','206일 표본 재현과 기존 모델 결과 재현을 구분'],
],columns=['항목','정의','상태','근거'])
save(requirements,'requirements');show(requirements)
theme_path=_input_file('raw/추가전달자료/2026-10-07/PBL과제 테마 재선정 및 적용방법 리스트_26.10.07.xlsx')
theme_wb=load_workbook(theme_path,read_only=True,data_only=False)
theme_cells=pd.DataFrame([{'source_file':theme_path.relative_to(ROOT).as_posix(),'sheet':ws.title,'cell':c.coordinate,'text':str(c.value)} for ws in theme_wb for row in ws for c in row if c.value is not None])
theme_wb.close();save(theme_cells,'supplement_theme_source_cells')
theme_reference=pd.DataFrame([
 ['테마-1','A7:A10','FRT/RR 대표품목','FRT 1품번, RR 2품번을 각각 추출','representative_chrome_events.csv'],
 ['테마-1','A12:A13 및 삽입 이미지','전류밀도·전압·시간','설정 전류밀도가 주지표, 실제평균은 보조지표. 이미지 산술 오류는 공식에 따라 계산','representative_process_statistics.csv'],
 ['테마-1','A14:A16','크롬산·철분·3가크롬·황산 및 A/B/TCC','성분과 첨가제 실측 농도 분석. 적용기간별 기준·공정-액조 대응은 미확정','solution_limits_summary.csv; additive_lab_coverage.csv'],
 ['테마-1','A17:A19','핀홀·피트와 공정·성분의 관계','검사 원본은 핀홀/피트 통합코드. 품번·LOT 부재로 날짜 탐색만 가능','lag_correlations.csv'],
 ['테마-1','E46:E50','불순물 상승 → 전압 상승, 밀도 감소·시간 증가 가설','설정 전류밀도 통제 및 실제평균 통제 민감도 비교. 인과·운영조건 확정 불가','field_feedback_revalidation.csv'],
 ['테마-1','Q3 인근 삽입 [예시] 표','품목별 일자 공정조건+도금액+불량 통합표의 예시','7/1~7/7 숫자는 [예시]로 표시된 설명 자료. 실제 관측/검사/두께 데이터나 확정 판정기준으로 사용하지 않음','supplement_theme_review.csv'],
 ['테마-2','A8:A11','에칭·도금 크롬산과 첨가제 투입량/전력량','월별 순투입/자동투입 대비 kWh와 Ah를 구분. 첨가제는 Nr.6 단일 투입계열','monthly_chemical_energy_ratios.csv; monthly_theme2_summary.csv'],
 ['테마-2','A8, A12','첨가제 2종 구분 및 적정 투입기준','로그에서는 Nr.6 한 계열만 구분되며 A/B/TCC와의 제품 대응은 미확정','monthly_theme2_summary.csv'],
],columns=['시트','원본 위치','요구사항','반영 및 한계','근거 파일'])
save(theme_reference,'supplement_theme_review')

## 3. 자료의 기간과 기록 수

자료 종류별로 파일 수, 기록 수와 수집 기간을 확인합니다. 서식에만 있는 빈 행은 실제 기록과 구분합니다.


In [ ]:
originals=sorted(p for p in (ROOT/'raw').rglob('*') if p.is_file())
inventory=pd.DataFrame([{'source_file':p.relative_to(ROOT).as_posix(),'bytes':p.stat().st_size,'sha256':digest(p)} for p in originals])
initial_hashes=dict(zip(inventory.source_file,inventory.sha256))
save(inventory,'source_files_sha256')
duplicates=[]
for copy_dir_name in ['도금액분석기록','생산작업이력','약품투입기록','월별 불량현황','일별 불량현황','전력사용기록']:
    p=ROOT/copy_dir_name
    if p.is_dir():
        for f in p.rglob('*'):
            if f.is_file():
                h=digest(f); same=inventory[inventory.sha256.eq(h)]
                duplicates.append({'copy':f.relative_to(ROOT).as_posix(),'canonical':';'.join(same.source_file),'hash_equal':len(same)>0})
duplicates=pd.DataFrame(duplicates,columns=['copy','canonical','hash_equal'])
save(duplicates,'duplicate_copies')
input_fingerprints={p.relative_to(ROOT).as_posix():digest(p) for p in sorted((ROOT/'csv').rglob('*.csv'))+sorted((ROOT/'prepared').rglob('*.csv'))+([ROOT/'한국로텍 실행과제.txt'] if (ROOT/'한국로텍 실행과제.txt').is_file() else [])}
schema=[]
for _,r in manifest.iterrows():
    a=rows_for(ROOT/r.csv_relative_path) if r.source_group!='production_history' else None
    if a is None:
        with (ROOT/r.csv_relative_path).open(encoding='utf-8-sig',newline='') as f: a=[next(csv.reader(f))]
    labels=a[0] if r.source_group=='production_history' else [v for row in a[:40] for v in row[:8] if v and not isdate(v)]
    schema.append({'group':r.source_group,'source_file':r.source_relative_path,'table':r.table_name,'csv':r.csv_relative_path,'converted_rows':r.row_count,'converted_columns':r.column_count,'key_labels':' ; '.join(dict.fromkeys(labels))[:1600]})
schema=pd.DataFrame(schema);save(schema,'source_tables_and_columns')
show(manifest.groupby('source_group').agg(files=('source_relative_path','nunique'),tables=('csv_relative_path','size'),converted_rows=('row_count','sum')))
print(f'원본 파일 {len(inventory):,}개를 확인했습니다.')

## 4. 날짜별 검사 결과

날짜별 검사수량과 불량기록을 집계합니다. 검사수량이 비어 있는 날은 비율을 계산하지 않습니다.

표의 **PPHV**는 `불량기록 건수 ÷ 검사수량 × 100`입니다. 한 제품에 여러 불량이 기록될 수 있으므로 불량 제품의 비율과는 다를 수 있습니다.


In [ ]:
daily_rows, defect_rows, daily_audit, raw_checks=[],[],[],[]
for _,r in manifest[(manifest.source_group=='daily_defects')&manifest.table_name.eq('구간 1')].iterrows():
    a=rows_for(ROOT/r.csv_relative_path)
    yy,mm=map(int,re.search(r'(20\d{2})년_(\d{2})월',r.csv_relative_path).groups())
    h=next(i for i,row in enumerate(a) if row[0]=='Problem Description')
    total=next(i for i,row in enumerate(a[h+1:],h+1) if row[0]=='Total')
    reviewed=next(i for i,row in enumerate(a[h+1:],h+1) if row[0]=='Total Reviewed')
    pphvrow=next(i for i,row in enumerate(a[h+1:],h+1) if row[0]=='PPHV')
    excluded=[]; current=[]
    for j,v in enumerate(a[h]):
        if not isdate(v):continue
        date=pd.Timestamp(v).normalize()
        if date.year!=yy or date.month!=mm: excluded.append(str(date.date()));continue
        ds=[]
        for i in range(h+1,total):
            val=0. if a[i][j].strip()=='' else num(a[i][j])
            defect_rows.append({'date':date,'defect_type':a[i][0],'count':val,'source_file':r.source_relative_path,'source_row':i+1,'source_col':j+1})
            ds.append(val)
        n=num(a[reviewed][j]); d=num(a[total][j]); p=num(a[pphvrow][j])
        daily_rows.append({'date':date,'total_defects':d,'total_reviewed':n,'pphv_source':p,'pphv':d/n*100 if n>0 else np.nan,'defect_type_sum':sum(ds),'source_file':r.source_relative_path,'source_col':j+1})
        current.append(j)
    daily_audit.append({'month':f'{yy}-{mm:02d}','title':a[1][6],'date_columns':len(current),'excluded_other_month_dates':';'.join(excluded),'sheet_total':num(a[total][5]),'date_column_total':sum(num(a[total][j]) for j in current),'sheet_reviewed':num(a[reviewed][5]),'date_column_reviewed':float(np.nansum([num(a[reviewed][j]) for j in current])),'missing_reviewed_dates':sum(pd.isna(num(a[reviewed][j])) for j in current),'formula_error_cells':sum(v.startswith(('#REF!','#DIV/0!','#VALUE!')) for row in a for v in row)})
    # 선택한 날짜의 숫자를 원본 셀과 대조합니다.
    wb=load_workbook(ROOT/r.source_relative_path,read_only=True,data_only=True)
    ws=wb['구간 1']; vals=list(ws.values); mismatch=0; tested=0
    for i in [h,*range(h+1,total),total,reviewed,pphvrow]:
        for j in current:
            x=vals[i][j] if j<len(vals[i]) else None; y=a[i][j];tested+=1
            if isinstance(x,datetime): ok=pd.Timestamp(x)==pd.Timestamp(y)
            elif x is None: ok=y==''
            elif isinstance(x,(int,float)): ok=np.isclose(float(x),num(y),equal_nan=True)
            else:ok=str(x)==y
            mismatch+=not ok
    raw_checks.append({'source_file':r.source_relative_path,'cells_checked':tested,'mismatches':mismatch});wb.close()
daily=pd.DataFrame(daily_rows).sort_values('date').reset_index(drop=True)
defects=pd.DataFrame(defect_rows)
pivot=defects.pivot(index='date',columns='defect_type',values='count')
daily=daily.merge(pivot[['도금-핀홀/피트','도금두께미달']],on='date',validate='one_to_one').rename(columns={'도금-핀홀/피트':'pinhole_pit_count','도금두께미달':'thickness_under_count'})
for target in ['pinhole_pit','thickness_under']:
    daily[target+'_pphv']=daily[target+'_count']/daily.total_reviewed.where(daily.total_reviewed>0)*100
daily['month']=daily.date.dt.strftime('%Y-%m')
valid=daily[daily.total_reviewed.gt(0)].copy()
prepared=pd.read_csv(ROOT/'prepared/five_fold/daily_targets.csv',parse_dates=['date'])
repro=daily.merge(prepared,on='date',suffixes=('_rebuilt','_prepared'),validate='one_to_one')
repro['count_difference']=repro.total_defects_rebuilt-repro.total_defects_prepared
repro['reviewed_difference']=repro.total_reviewed_rebuilt-repro.total_reviewed_prepared
repro['valid_denominator']=repro.total_reviewed_rebuilt.gt(0)
repro['prepared_zero_for_source_blank']=repro.total_reviewed_rebuilt.isna()&repro.total_reviewed_prepared.eq(0)
save(daily,'daily_inspection_rebuilt');save(defects,'defect_counts_rebuilt');save(pd.DataFrame(daily_audit),'inspection_sheet_audit');save(pd.DataFrame(raw_checks),'original_cell_checks');save(repro,'reproduction_comparison')
monthly=daily.groupby('month').agg(calendar_days=('date','size'),valid_days=('total_reviewed',lambda s:int((s>0).sum())),defects=('total_defects','sum'),reviewed=('total_reviewed','sum'),pinhole_pit=('pinhole_pit_count','sum'),thickness_under=('thickness_under_count','sum')).reset_index()
monthly['total_pphv']=100*monthly.defects/monthly.reviewed;monthly['pinhole_pit_pphv']=100*monthly.pinhole_pit/monthly.reviewed
save(monthly,'monthly_inspection');show(monthly)
print(f'검사 기록 {len(daily)}일 중 검사수량이 있는 {len(valid)}일의 비율을 계산했습니다.')

## 5. 생산·도금 작업 기록

분석 대상 품목의 전류·전압·온도·처리시간을 확인합니다.
같은 제품이 공정을 다시 통과할 수 있어 기록 수가 제품 수와 같지는 않습니다.


In [ ]:
chrome_frames,production_audit,route_frames=[],[],[]
for _,r in manifest[manifest.source_group=='production_history'].iterrows():
    raw_count=0; stations={}; nitems=set(); exact_dupes=0; entirely_blank=0; station_missing_nonempty=0; valid_time_min=None; valid_time_max=None
    for chunk in pd.read_csv(ROOT/r.csv_relative_path,dtype=str,keep_default_na=False,chunksize=100000):
        cols=list(chunk.columns); raw_count+=len(chunk)
        blank=chunk.eq('').all(axis=1); entirely_blank+=int(blank.sum())
        station_missing_nonempty+=int((chunk.iloc[:,2].eq('') & ~blank).sum())
        times=pd.to_datetime(chunk.iloc[:,1],errors='coerce')
        if times.notna().any():
            valid_time_min=times.min() if valid_time_min is None else min(valid_time_min,times.min())
            valid_time_max=times.max() if valid_time_max is None else max(valid_time_max,times.max())
        for st,n in chunk.iloc[:,2].value_counts().items():stations[st]=stations.get(st,0)+int(n)
        nitems.update(chunk.iloc[:,7].unique())
        chosen=chunk[chunk.iloc[:,7].isin(ITEMS)]
        if len(chosen):
            route_frames.append(pd.DataFrame({'item':chosen.iloc[:,7],'fb_key':chosen.iloc[:,3].map(lambda x:parts(x)[1] if len(parts(x))>1 else ''),'station':chosen.iloc[:,2],'source_file':r.source_relative_path}))
        c=chunk[chunk.iloc[:,2].str.contains('크롬',regex=False)].copy()
        if c.empty:continue
        z=pd.DataFrame({'start_time':pd.to_datetime(c.iloc[:,0],errors='coerce'),'end_time':pd.to_datetime(c.iloc[:,1],errors='coerce'),'station':c.iloc[:,2],'fb_raw':c.iloc[:,3],'item':c.iloc[:,7].str.strip(),'item_name':c.iloc[:,8],'parts_count':pd.to_numeric(c.iloc[:,9],errors='coerce'),'galvanic_area_dm2':pd.to_numeric(c.iloc[:,10],errors='coerce'),'geometric_area_dm2':pd.to_numeric(c.iloc[:,11],errors='coerce')})
        z['fb_key']=c.iloc[:,3].map(lambda x:parts(x)[1] if len(parts(x))>1 else '')
        for label,idx in [('current_a',13),('voltage_v',17),('temperature_c',19)]:
            for k,name in enumerate(['set','min','max','avg']):z[label+'_'+name]=c.iloc[:,idx].map(lambda v:slot(v,k))
        z['duration_set_s']=c.iloc[:,5].map(lambda v:sec(parts(v)[0]) if len(parts(v)) else np.nan)
        z['duration_actual_s']=c.iloc[:,5].map(lambda v:sec(parts(v)[1]) if len(parts(v))>1 else np.nan)
        z['duration_timestamp_s']=(z.end_time-z.start_time).dt.total_seconds()
        z['charge_actual_ah']=c.iloc[:,14].map(lambda v:slot(v,1))
        z['date']=z.end_time.dt.normalize()
        z['density_set_a_dm2']=z.current_a_set/z.geometric_area_dm2.where(z.geometric_area_dm2>0)
        z['density_actual_a_dm2']=z.current_a_avg/z.geometric_area_dm2.where(z.geometric_area_dm2>0)
        z['voltage_low_flag']=z.voltage_v_avg.le(VOLTAGE_FLOOR)&z.current_a_avg.gt(0)
        z['voltage_clean_v']=z.voltage_v_avg.mask(z.voltage_low_flag)
        z['source_file']=r.source_relative_path;z['source_row']=c.index+2
        chrome_frames.append(z)
    production_audit.append({'source_file':r.source_relative_path,'data_rows':raw_count,'part_numbers':len(nitems),'station_counts':json.dumps(stations,ensure_ascii=False),'entirely_blank_rows':entirely_blank,'station_missing_nonempty_rows':station_missing_nonempty,'station_present_rows':raw_count-stations.get('',0),'first_valid_end_time':valid_time_min,'last_valid_end_time':valid_time_max})
    print(f'{Path(r.source_relative_path).name}: {raw_count:,}행 확인')
chrome=pd.concat(chrome_frames,ignore_index=True)
dupekeys=['start_time','end_time','station','fb_raw','item']
duplicate_mask=chrome.duplicated(dupekeys,keep='first')
save(chrome[duplicate_mask],'production_exact_duplicates')
chrome=chrome[~duplicate_mask].copy()
representative=chrome[chrome.item.isin(ITEMS)].copy()
chrome['active_event']=chrome.current_a_avg.gt(0)&chrome.geometric_area_dm2.gt(0)&chrome.duration_actual_s.gt(0)
active_chrome=chrome[chrome.active_event].copy()
active_chrome['temperature_c_avg']=active_chrome.temperature_c_avg.where(active_chrome.temperature_c_avg>0)
active_representative=active_chrome[active_chrome.item.isin(ITEMS)].copy()
routes=pd.concat(route_frames,ignore_index=True)
save(pd.DataFrame(production_audit),'production_file_rows');save(representative,'representative_chrome_events')
route_stats=routes.groupby(['item','fb_key']).station.agg(['size','nunique']).reset_index()
save(route_stats,'representative_fb_route_summary')
process_cols=['density_set_a_dm2','density_actual_a_dm2','current_a_avg','voltage_v_avg','voltage_clean_v','temperature_c_avg','duration_actual_s','duration_set_s','duration_timestamp_s','charge_actual_ah']
item_summary=representative.groupby('item').agg(events=('item','size'),fb_ids=('fb_key','nunique'),first_date=('date','min'),last_date=('date','max'),days=('date','nunique'),logged_parts=('parts_count','sum'),low_voltage_events=('voltage_low_flag','sum')).reset_index()
save(item_summary,'representative_summary')
distributions=active_representative.groupby(['item','station'])[process_cols].agg(['count','mean','std','min','median','max'])
distributions.columns=['__'.join(c) for c in distributions.columns];save(distributions.reset_index(),'representative_process_statistics')
process_daily=active_chrome.groupby('date')[process_cols].mean()
process_daily['chrome_events']=chrome.groupby('date').size()
item_daily=active_representative.groupby(['date','item'])[process_cols].mean().reset_index()
save(process_daily.reset_index(),'process_daily_all_items');save(item_daily,'process_daily_representatives')
show(item_summary)

## 6. 도금액 성분

도금액의 성분 측정값과 원본에 적힌 규격·관리범위를 비교합니다.
같은 날짜에 서로 다른 값이 기록된 경우에는 관리치 시트의 값을 우선 사용하고, 나머지 값도 따로 보관합니다.


In [ ]:
solution_records=[]
components=['크롬산','3가크롬','철분','황산','이물질']
def bounds(v):
    ns=re.findall(r'\d+(?:\.\d+)?',str(v))
    return (float(ns[0]),float(ns[1])) if len(ns)==2 else (np.nan,np.nan)
for _,r in manifest[(manifest.source_group=='plating_solution')&manifest.table_name.isin(['농도','액관리대장','액관리대장(관리치)','액관리대장-6월','액관리대장-7월'])].iterrows():
    a=rows_for(ROOT/r.csv_relative_path); old=(r.table_name=='농도')
    priority={'농도':1,'액관리대장':2,'액관리대장(관리치)':3,'액관리대장-6월':4,'액관리대장-7월':4}[r.table_name]
    dates={}; line=None;bath=None
    for i,row in enumerate(a):
        if sum(isdate(v) for v in row)>5:
            dates={j:pd.Timestamp(v).normalize() for j,v in enumerate(row) if isdate(v)};line=None;continue
        if not dates:continue
        label=row[2] if old else row[0]
        mt=re.search(r'([134])라인',label)
        if mt:line=int(mt[1]);bath='에칭조' if '에칭' in label else '도금조' if '도금' in label else None
        component=row[3] if len(row)>3 else ''
        if line is None or bath is None or component not in components:continue
        slo,shi=(num(row[0]),num(row[1])) if old else bounds(row[1])
        mlo,mhi=(np.nan,np.nan) if old else bounds(row[2])
        for j,date in dates.items():
            value=num(row[j]) if j<len(row) else np.nan
            solution_records.append({'date':date,'line':line,'bath':bath,'component':component,'value_raw':value,'spec_low':slo,'spec_high':shi,'manage_low':mlo,'manage_high':mhi,'priority':priority,'source_file':r.source_relative_path,'source_sheet':r.table_name,'source_row':i+1,'source_col':j+1})
solution_all=pd.DataFrame(solution_records)
solkey=['date','line','bath','component']
conflict_keys=solution_all.groupby(solkey).value_raw.nunique().loc[lambda s:s>1].reset_index()[solkey]
conflicts=solution_all.merge(conflict_keys,on=solkey,how='inner')
save(conflicts,'solution_conflicting_sources')
# 중복된 측정값은 관리치 시트를 우선 사용합니다.
solution_all['populated']=solution_all.value_raw.notna()
solution=solution_all.sort_values(['populated','priority']).drop_duplicates(solkey,keep='last').copy()
anchor=solution[solution.component.eq('크롬산')][['date','line','bath','value_raw']].rename(columns={'value_raw':'chromic_acid_anchor'})
solution=solution.merge(anchor,on=['date','line','bath'],validate='many_to_one')
solution['unmeasured_group']=solution.chromic_acid_anchor.isna()|solution.chromic_acid_anchor.le(0)
solution['value']=solution.value_raw.mask(solution.unmeasured_group)
solution['outside_spec']=((solution.value<solution.spec_low)|(solution.value>solution.spec_high)).where(solution.value.notna()&solution.spec_low.notna()&solution.spec_high.notna())
solution['outside_management']=((solution.value<solution.manage_low)|(solution.value>solution.manage_high)).where(solution.value.notna()&solution.manage_low.notna()&solution.manage_high.notna())
save(solution,'solution_components_clean')
solution['feature']=solution.bath+'_L'+solution.line.astype(str)+'_'+solution.component
solution_wide=solution.pivot(index='date',columns='feature',values='value')
spec_summary=solution.groupby(['line','bath','component']).agg(measured=('value','count'),spec_evaluable=('outside_spec','count'),spec_exceed=('outside_spec','sum'),management_evaluable=('outside_management','count'),management_exceed=('outside_management','sum')).reset_index()
save(spec_summary,'solution_limits_summary')
lab=[]
for _,r in manifest[(manifest.source_group=='plating_solution')&manifest.table_name.eq('아토텍-분석자료')].iterrows():
    a=rows_for(ROOT/r.csv_relative_path);dates={};last=None;line=None;bath=None
    for j,v in enumerate(a[2]):
        if isdate(v):last=pd.Timestamp(v).normalize()
        if last is not None:dates[j]=last
    for i,row in enumerate(a[4:],4):
        mt=re.search(r'([134])라인',row[0])
        if mt:line=int(mt[1]);bath='에칭조' if '에칭' in row[0] else '도금조'
        if bath!='도금조' or not any(t in row[3] for t in ['ADDITIVE','Additive','TCC']):continue
        for j,date in dates.items():
            if j<len(row) and a[3][j]=='아토텍' and pd.notna(num(row[j])):
                lab.append({'date':date,'line':line,'component':row[3],'value':num(row[j]),'unit':row[2],'range':row[1],'source_file':r.source_relative_path,'source_row':i+1,'source_col':j+1})
lab=pd.DataFrame(lab);save(lab,'additive_lab_measurements')
print(f'성분 측정 {solution.value.notna().sum():,}건, 첨가제 측정 {len(lab):,}건을 확인했습니다. 서로 다른 값이 기록된 항목 {len(conflict_keys):,}건은 별도 표에 저장했습니다.')
show(spec_summary[spec_summary.bath.eq('도금조')])

## 7. 약품 사용량과 전력량

날짜별 약품 투입량을 더해 월별 합계와 맞는지 확인하고, 전력량과 함께 비교합니다.
**kWh**는 사용한 전기에너지, **Ah**는 전류가 흐른 양을 나타냅니다.


In [ ]:
chem_daily_rows,chem_audits,chem_events=[],[],[]
for _,r in manifest[manifest.source_group=='chemical_inputs'].iterrows():
    a=rows_for(ROOT/r.csv_relative_path)
    nr={'에칭':1,'도금':2,'첨가제':6}[next(k for k in ['에칭','도금','첨가제'] if k in r.csv_relative_path)]
    kind={1:'etch_chromic',2:'chrome_chromic',6:'additive'}[nr]
    day_entries=[];event_entries=[]
    dh=next(row for row in a if row[0]=='날짜 시간')
    daynr=dh.index('Nr.'); dm=next(j for j,v in enumerate(dh) if v.startswith('손 [ml]'))
    da=next(j for j,v in enumerate(dh) if '자동적' in v);dx=next(j for j,v in enumerate(dh) if v.startswith('수동 [ml]'));dc=next(j for j,v in enumerate(dh) if 'Ah/dm' in v)
    eh=next(row for row in a if row[0]=='Nr.' and '날짜 시간' in row)
    et=eh.index('날짜 시간');ev=next(j for j,v in enumerate(eh) if ('총액' in v or '합계 [ml]' in v))
    mh=next(i for i,row in enumerate(a) if row[0]=='Nr.' and '날짜 시간' not in row)
    mt=next(j for j,v in enumerate(a[mh]) if v.startswith('합계 [ml]'))
    for i,row in enumerate(a):
        if isdate(row[0]) and num(row[daynr])==nr:
            entry={'date':pd.Timestamp(row[0]).normalize(),'chemical':kind,'manual_ml':num(row[dm]),'automatic_ml':num(row[da]),'additional_manual_ml':num(row[dx]),'counter_ah_dm2':num(row[dc]),'source_file':r.source_relative_path,'source_row':i+1}
            entry['net_ml']=entry['manual_ml']+entry['automatic_ml']+entry['additional_manual_ml'];day_entries.append(entry)
        if num(row[0])==nr and isdate(row[et]):
            event_entries.append({'date':pd.Timestamp(row[et]).normalize(),'chemical':kind,'event_total_ml':num(row[ev]),'source_file':r.source_relative_path})
    chem_daily_rows.extend(day_entries);chem_events.extend(event_entries)
    monthrow=next(row for row in a[mh+1:mh+6] if num(row[0])==nr)
    chem_audits.append({'source_file':r.source_relative_path,'chemical':kind,'daily_rows':len(day_entries),'events':len(event_entries),'monthly_net_ml':num(monthrow[mt]),'daily_net_sum_ml':sum(x['net_ml'] for x in day_entries),'event_total_sum_ml':sum(x['event_total_ml'] for x in event_entries),'negative_manual_days':sum(x['manual_ml']<0 for x in day_entries)})
chem=pd.DataFrame(chem_daily_rows).sort_values(['date','chemical'])
chem_audit=pd.DataFrame(chem_audits);chem_audit['monthly_minus_daily']=chem_audit.monthly_net_ml-chem_audit.daily_net_sum_ml;chem_audit['daily_minus_events']=chem_audit.daily_net_sum_ml-chem_audit.event_total_sum_ml
save(chem,'chemical_daily');save(chem_audit,'chemical_reconciliation')
chem_wide=chem.pivot(index='date',columns='chemical',values=['net_ml','automatic_ml'])
chem_wide.columns=['dose_'+b+'_'+a for a,b in chem_wide.columns]
power=[]
for _,r in manifest[manifest.source_group=='power_usage'].iterrows():
    a=rows_for(ROOT/r.csv_relative_path);month=pd.Timestamp(a[3][7]).strftime('%Y-%m')
    for i,row in enumerate(a):
        st=num(row[0])
        if st in [13,14,15,16]:power.append({'month':month,'station':int(st),'charge_ah':num(row[6]),'primary_kwh':num(row[11]),'secondary_kwh':num(row[18]),'source_file':r.source_relative_path,'source_row':i+1})
power=pd.DataFrame(power);save(power,'power_monthly_stations')
cm=chem.assign(month=chem.date.dt.strftime('%Y-%m')).groupby(['month','chemical'])[['net_ml','automatic_ml']].sum().reset_index()
pm=power.assign(chemical=np.where(power.station.eq(13),'etch_chromic','chrome_chromic')).groupby(['month','chemical'])[['charge_ah','primary_kwh','secondary_kwh']].sum().reset_index()
energy=cm.merge(pm,on=['month','chemical'],how='left',validate='one_to_one')
energy['net_ml_per_1000ah']=energy.net_ml/energy.charge_ah*1000
energy['net_ml_per_kwh']=energy.net_ml/energy.primary_kwh
energy['automatic_ml_per_1000ah']=energy.automatic_ml/energy.charge_ah*1000
energy['automatic_ml_per_kwh']=energy.automatic_ml/energy.primary_kwh
save(energy,'monthly_chemical_energy_ratios')
theme2_monthly=energy[energy.chemical.ne('additive')].pivot(index='month',columns='chemical',values=['net_ml_per_kwh','automatic_ml_per_kwh'])
theme2_monthly.columns=[a+'__'+b for a,b in theme2_monthly.columns]
theme2_additive=energy[energy.chemical.eq('additive')].set_index('month')[['net_ml','automatic_ml']].add_prefix('additive_')
theme2_monthly=theme2_monthly.join(theme2_additive).reset_index()
save(theme2_monthly,'monthly_theme2_summary')
show(chem_audit[['chemical','daily_rows','events','monthly_minus_daily','daily_minus_events']].groupby('chemical').agg(['sum','max']))

## 8. 생산 기록과 검사 기록 연결

검사자료에는 날짜별 합계만 있고 품번이나 생산묶음 번호(LOT)는 없습니다.
따라서 같은 제품을 직접 연결할 수는 없으며, 날짜를 기준으로 생산 기록과 검사 결과를 비교합니다.


In [ ]:
features=process_daily.copy()
for item in ITEMS:
    z=item_daily[item_daily.item.eq(item)].set_index('date')[['density_set_a_dm2','density_actual_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s']]
    z.columns=[item+'__'+c for c in z.columns];features=features.join(z,how='outer')
features=features.join(solution_wide,how='outer').join(chem_wide,how='outer')
lab['feature']='lab_L'+lab.line.astype(str)+'_'+lab.component
features=features.join(lab.pivot(index='date',columns='feature',values='value'),how='outer')
for c in chem_wide.columns:
    features[c+'_per_100_events']=features[c]/features.chrome_events.where(features.chrome_events>0)*100
save(features.reset_index(),'daily_features')
join_rows=[{'join':'품번+LOT: 생산→도금→검사','denominator_name':'기업 LOT 모집단(미제공)','denominator':np.nan,'matched':0,'rate':np.nan,'reason':'완전 연결을 확인한 기록 0건. LOT 분모 미정이므로 연결률 산출 불가. 검사 206일을 LOT 분모로 대체하지 않음'},
 {'join':'대표품목 공정4조건 값 존재','denominator_name':'대표품목 크롬 통과행','denominator':len(representative),'matched':int(representative[['density_actual_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s']].notna().all(axis=1).sum()),'reason':'비결측 여부만 확인. 0 또는 비활성 기록을 포함하므로 유효 표본과 구분'},
 {'join':'대표품목 공정4조건 유효','denominator_name':'대표품목 크롬 통과행','denominator':len(representative),'matched':int(active_representative[['density_actual_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s']].gt(0).all(axis=1).sum()),'reason':'활성 공정 및 전류·면적·시간·온도 양수, 전압>0.1V 진단 가정'}]
for lag in LAGS:
    dates=valid.date-pd.to_timedelta(lag,unit='D')
    f=features.reindex(dates).reset_index(drop=True)
    for desc,cs in [('전체품목 공정 일자',['density_set_a_dm2','temperature_c_avg','duration_actual_s']),('약품 일자',list(chem_wide.columns)),('도금액 1라인 성분',list(solution_wide.filter(regex='도금조_L1_').columns))]:
        match=f[cs].notna().all(axis=1)
        join_rows.append({'join':f'{desc} 시차{lag}일','denominator_name':'유효 일별 검사 레코드','denominator':len(valid),'matched':int(match.sum()),'reason':'날짜 수준 연결. 개별 제품 노출 또는 LOT 연결 아님'})
    for item in ITEMS:
        cs=[item+'__'+c for c in ['density_set_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s']]
        join_rows.append({'join':f'{item} 공정4조건 시차{lag}일','denominator_name':'유효 일별 검사 레코드','denominator':len(valid),'matched':int(f[cs].notna().all(axis=1).sum()),'reason':'해당 품번 공정 존재일과 전체품목 검사일 연결. 해당 품번 불량률 아님'})
joins=pd.DataFrame(join_rows);joins['rate']=joins.matched/joins.denominator
save(joins,'join_coverage');show(joins)
key_audit=pd.DataFrame([
 ['생산작업이력','Part- 번호','있음','대표 3품번 정확 일치'],['생산작업이력','FB- Ident no.','있음','설비 운행 ID. 기업 LOT와 대응표 없음'],['생산작업이력','할당','마스킹','****************'],
 ['일별 검사','품번·LOT·라인','없음','날짜/불량유형/검사수량 전체 합계'],['월별 보고','품명','부분 있음','WORST TOP5 등 선택 표본. 모든 품목 검사분모 없음'],['액분석','라인1/3/4','있음','생산 스테이션14/15/16 대응 미확정'],['약품투입','스테이션13,14~16','있음','검사 LOT는 없음']
],columns=['자료','연결키','상태','판정']);save(key_audit,'key_availability')
missing_target=daily[~daily.total_reviewed.gt(0)].merge(process_daily[['chrome_events']].reset_index(),on='date',how='left')
save(missing_target,'missing_reviewed_days_with_production');show(missing_target[['date','total_reviewed','chrome_events']])

## 9. 검사일과 작업일 비교

검사 결과를 **같은 날, 5일 전, 10일 전**의 작업 기록과 각각 비교합니다.
표의 **상관계수(r)**는 두 값이 함께 변하는 정도입니다. 양수이면 함께 커지는 경향, 음수이면 한쪽이 커질 때 다른 쪽은 작아지는 경향을 뜻합니다.
이 결과만으로 불량의 원인을 확정할 수는 없습니다.


In [ ]:
def bh(p):
    p=np.asarray(p,float);result=np.full(len(p),np.nan);ok=np.isfinite(p)
    order=np.argsort(p[ok]);vals=p[ok][order];adj=np.minimum.accumulate((vals*len(vals)/np.arange(1,len(vals)+1))[::-1])[::-1]
    pos=np.flatnonzero(ok)[order];result[pos]=np.minimum(adj,1);return result
def residual_corr(x,y,controls):
    xr=stats.rankdata(x);yr=stats.rankdata(y)
    c=np.column_stack([np.ones(len(x)),np.asarray(controls,float)])
    rx=xr-c@np.linalg.lstsq(c,xr,rcond=None)[0];ry=yr-c@np.linalg.lstsq(c,yr,rcond=None)[0]
    return float(np.corrcoef(rx,ry)[0,1]) if np.std(rx)>1e-10 and np.std(ry)>1e-10 else np.nan
lag_matrices={lag:features.reindex(valid.date-pd.to_timedelta(lag,unit='D')).set_axis(valid.index) for lag in LAGS}
targets=['pphv','pinhole_pit_pphv','thickness_under_pphv']
corr_rows=[]
for feature in features.columns:
    common=np.logical_and.reduce([lag_matrices[k][feature].notna().to_numpy() for k in LAGS])
    for lag in LAGS:
        for target in targets:
            for sample in ['available','common_3_lags']:
                mask=lag_matrices[lag][feature].notna()&valid[target].notna()
                if sample=='common_3_lags':mask &= common
                x=lag_matrices[lag].loc[mask,feature];y=valid.loc[mask,target];n=len(x)
                rr=pp=partial=np.nan
                if n>=20 and x.nunique()>1 and y.nunique()>1:
                    rr,pp=stats.spearmanr(x,y)
                    controls=pd.get_dummies(pd.DataFrame({'month':valid.loc[mask,'month'],'weekday':valid.loc[mask,'date'].dt.dayofweek.astype(str)}),drop_first=True,dtype=float)
                    controls['log_reviewed']=np.log1p(valid.loc[mask,'total_reviewed'])
                    controls['log_process_events']=np.log1p(lag_matrices[lag].loc[mask,'chrome_events']).fillna(0)
                    partial=residual_corr(x,y,controls)
                corr_rows.append({'feature':feature,'lag_days':lag,'target':target,'sample':sample,'n_days':n,'spearman_r':rr,'p_nominal':pp,'month_weekday_volume_adjusted_r':partial,'unit_of_analysis':'all-item inspection day'})
correlations=pd.DataFrame(corr_rows)
correlations['q_bh']=correlations.groupby(['target','sample']).p_nominal.transform(bh)
save(correlations,'lag_correlations')
show(correlations.query("target=='pinhole_pit_pphv' and sample=='common_3_lags'").assign(abs_r=lambda d:d.spearman_r.abs()).sort_values('abs_r',ascending=False).head(15))

## 10. 도금액 성분과 작업조건 비교

도금액 성분이 달라질 때 전압·전류밀도·처리시간도 함께 달라지는지 살펴봅니다.
생산설비와 도금액을 담는 탱크의 연결 정보가 없어 날짜 단위로 비교합니다.


In [ ]:
sensor=active_chrome.groupby('station').agg(events=('station','size'),low_voltage=('voltage_low_flag','sum'),voltage_raw_median=('voltage_v_avg','median'),voltage_clean_median=('voltage_clean_v','median'),current_median=('current_a_avg','median'),temperature_median=('temperature_c_avg','median')).reset_index()
sensor['low_voltage_fraction']=sensor.low_voltage/sensor.events
save(sensor,'station_sensor_diagnostics');show(sensor)
feedback=[]
panel=active_representative.groupby(['date','item','station'])[process_cols].mean().reset_index()
for (item,station),g in panel.groupby(['item','station']):
    g=g.merge(solution_wide.reset_index(),on='date',how='left',validate='many_to_one')
    for line in [1,3,4]:
        for comp in ['3가크롬','철분','이물질']:
            feature=f'도금조_L{line}_{comp}'
            for outcome in ['voltage_clean_v','density_set_a_dm2','duration_actual_s']:
                needed=[feature,outcome,'temperature_c_avg']
                if outcome=='voltage_clean_v':needed+=['density_set_a_dm2','density_actual_a_dm2','duration_actual_s']
                s=g.dropna(subset=needed)
                rr=partial=partial_actual=np.nan
                if len(s)>=30 and s[feature].nunique()>1 and s[outcome].nunique()>1:
                    rr=stats.spearmanr(s[feature],s[outcome]).statistic
                    controls=pd.get_dummies(s.date.dt.strftime('%Y-%m'),drop_first=True,dtype=float)
                    controls['temperature']=s.temperature_c_avg
                    if outcome=='voltage_clean_v':controls['density']=s.density_set_a_dm2;controls['duration']=s.duration_actual_s
                    partial=residual_corr(s[feature],s[outcome],controls)
                    if outcome=='voltage_clean_v':
                        actual_controls=controls.copy();actual_controls['density']=s.density_actual_a_dm2
                        partial_actual=residual_corr(s[feature],s[outcome],actual_controls)
                feedback.append({'item':item,'station':station,'chemical_line':line,'component':comp,'outcome':outcome,'n_days':len(s),'spearman_r':rr,'controlled_rank_r':partial,'voltage_actual_density_sensitivity_r':partial_actual,'control_density':'set_current/area' if outcome=='voltage_clean_v' else 'not included','exposure_mapping':'unconfirmed; date association only'})
feedback=pd.DataFrame(feedback);save(feedback,'field_feedback_revalidation')
thickness=pd.DataFrame([{'calendar_days':len(daily),'valid_days':len(valid),'positive_days':int((valid.thickness_under_count>0).sum()),'defect_count':float(valid.thickness_under_count.sum()),'representative_thickness_measurements':0,'other_part_thickness_records':21,'thickness_unit_confirmed':False,'part_lot_labeled_samples':0,'interpretation':'다른 6품번 21값·8조건·1일. 단위·측정위치·개체 미확정. 대표품목 두께/LOT 예측·공정능력 불가'}])
save(thickness,'thickness_feasibility');show(thickness)
lab_coverage=lab.groupby(['line','component']).agg(measurements=('value','size'),first_date=('date','min'),last_date=('date','max')).reset_index()
lab_coverage['same_day_inspection_days']=lab.groupby(['line','component']).apply(lambda g:g[g.date.isin(valid.date)].date.nunique(),include_groups=False).values
save(lab_coverage,'additive_lab_coverage')
temp_rows=[]
for _,r in manifest[(manifest.source_group=='plating_solution')&manifest.table_name.eq('액온도-측정자료')].iterrows():
    a=rows_for(ROOT/r.csv_relative_path);line=None;controllers={}
    for i,row in enumerate(a):
        mt=re.search(r'([134])라인',row[0])
        if mt:line=int(mt[1])
        if line is None:continue
        if row[2].startswith('Controller'):controllers[line]=row
        if row[2].startswith('실측'):
            for j,v in enumerate(row):
                if j<len(a[3]) and isdate(a[3][j]) and pd.notna(num(v)):
                    cv=num(controllers.get(line,['']*len(row))[j]);temp_rows.append({'date':pd.Timestamp(a[3][j]).normalize(),'line':line,'tank':row[2],'measured_c':num(v),'controller_c':cv,'measured_minus_controller_c':num(v)-cv})
temperature_compare=pd.DataFrame(temp_rows);save(temperature_compare,'temperature_controller_comparison')

## 11. 이미지와 측정값 확인

기존에 정리한 이미지 기록과 원본의 두께·균열·이온교환 측정값을 대조합니다.
다른 품목의 두께 기록은 분석 대상 품목의 측정값에 포함하지 않습니다.


In [ ]:
image_summary=json.loads((IMAGE_AUDIT/'summary.json').read_text(encoding='utf-8'))
expected_inputs=pd.read_csv(IMAGE_AUDIT/'05_재실행도구/analysis_input_sha256.csv')
assert all(digest(_input_file(row.file))==row.sha256 for row in expected_inputs.itertuples()), '고정 입력과 다른 파일: 변환CSV와 이미지검토를 먼저 갱신하세요.'
image_tables={p.stem:pd.read_csv(p) for p in (IMAGE_AUDIT/'03_검토표').glob('*.csv')}
measurement_source=next((ROOT/'raw/도금액분석기록').rglob('*2026년.xlsx'))
wb_measure=load_workbook(measurement_source,read_only=False,data_only=True)
source_checks=[]
for name in ['두께_실측기록','마이크로크랙_시험기록']:
    frame=image_tables[name].copy()
    for row in frame.itertuples():
        ws=wb_measure[row.source_sheet]
        actual=ws[row.source_cell].value
        same=bool(np.isclose(actual,row.value))
        date_value=ws[row.context_date_cell].value
        same=same and pd.Timestamp(date_value).normalize()==pd.Timestamp(row.date)
        same=same and str(ws[row.context_part_cell].value)==str(row.part_number)
        actual_spec=ws[row.spec_cell].value
        same=same and ((actual_spec is None and pd.isna(row.spec_source)) or str(actual_spec)==str(row.spec_source))
        source_checks.append({'source_sheet':row.source_sheet,'source_cell':row.source_cell,'value':actual,'passed':same})
    save(frame,'image_'+name)
ion=image_tables['이온교환_원본셀'].copy()
for i,row in ion.iterrows():
    ws=wb_measure[row.source_sheet]
    for col,field in [('before_cell','before'),('after_cell','after')]:
        actual=num(ws[row[col]].value)
        source_checks.append({'source_sheet':row.source_sheet,'source_cell':row[col],'value':actual,'passed':bool(np.isclose(actual,row[field],equal_nan=True))})
        ion.at[i,field]=actual
ion['computed_efficiency_percent']=100*(ion.before-ion.after)/ion.before.where(ion.before>0)
save(ion,'image_이온교환_원본셀')
wb_measure.close()
image_cell_checks=pd.DataFrame(source_checks);save(image_cell_checks,'image_source_cell_checks')
assert image_cell_checks.passed.all(), '이미지 검토 숫자와 현재 원본 셀 불일치'
for name,frame in image_tables.items():
    if name not in ['두께_실측기록','마이크로크랙_시험기록','이온교환_원본셀']:
        save(frame,'image_'+name)
thickness_records=image_tables['두께_실측기록']
assert len(thickness_records)==21 and not thickness_records.is_representative.any()
assert image_summary['files_checked']==len(_PREPARED_INPUTS['review_inventory']) and image_summary['unique_image_assets']==140
assert len(image_tables['이미지별_분류'])==140
show(thickness_records[['date','part_number','source_sheet','source_cell','value','unit','spec_source']])
show(ion[['date','component','before','after','computed_efficiency_percent','threshold_comparison']])

## 12. 월별 보고서와 출하검사 비교

월별 보고서는 생산공정의 불량을, 출하검사 자료는 출하 시점의 검사 결과를 담고 있습니다.
서로 집계 대상이 달라 두 결과를 나란히 확인합니다.


In [ ]:
ns={'a':'http://schemas.openxmlformats.org/drawingml/2006/main'}
ppt_tables,ppt_monthly=[],[]
for p in sorted((ROOT/'raw'/'월별불량현황').rglob('*.pptx')):
    yy,mm=map(int,re.search(r'_(\d{2})년\s*(\d+)월',p.name).groups());yy+=2000
    with zipfile.ZipFile(p) as z:
        for sn in sorted(n for n in z.namelist() if re.match(r'ppt/slides/slide\d+\.xml$',n)):
            e=ET.fromstring(z.read(sn))
            for ti,tbl in enumerate(e.findall('.//a:tbl',ns),1):
                arr=[[''.join(t.text or '' for t in cell.findall('.//a:t',ns)) for cell in row.findall('a:tc',ns)] for row in tbl.findall('a:tr',ns)]
                for ri,row in enumerate(arr):
                    for ci,v in enumerate(row):ppt_tables.append({'source_file':p.relative_to(ROOT).as_posix(),'slide':sn,'table':ti,'row':ri+1,'column':ci+1,'text':v})
                labels={re.sub(r'\s+','',row[0]):row for row in arr if row}
                if '생산수량' not in labels or '불량수량' not in labels or not any('실적(ppm)' in k.lower() for k in labels):continue
                header=next((row for row in arr if f'{mm}월' in [v.strip() for v in row]),None)
                if header is None:continue
                j=[v.strip() for v in header].index(f'{mm}월')
                actual=next(row for k,row in labels.items() if '실적(ppm)' in k.lower())
                ppt_monthly.append({'month':f'{yy}-{mm:02d}','defects_ppt':num(labels['불량수량'][j]),'production_ppt':num(labels['생산수량'][j]),'ppm_reported':num(actual[j]),'source_file':p.relative_to(ROOT).as_posix(),'slide':sn})
save(pd.DataFrame(ppt_tables),'monthly_ppt_tables_long')
ppt_monthly=pd.DataFrame(ppt_monthly).drop_duplicates(['month','source_file','slide'])
ppt_monthly['ppm_recalculated']=ppt_monthly.defects_ppt/ppt_monthly.production_ppt*1e6
save(ppt_monthly,'monthly_ppt_overall')
monthly_comparison=monthly.merge(ppt_monthly,on='month',how='left',validate='one_to_many')
save(monthly_comparison,'inspection_vs_monthly_ppt');show(monthly_comparison[['month','defects','reviewed','total_pphv','defects_ppt','production_ppt','ppm_reported']])

## 13. 예측 결과 확인

과거 자료로 두 예측 방법(Ridge, Random Forest)을 학습한 뒤, 나중 날짜의 실제 결과와 비교합니다.
표의 **MAE**는 예측과 실제 값의 차이를 평균한 값으로, 작을수록 예측이 잘 맞습니다.
예를 들어 실제 비율 5%를 7%로 예측했다면 차이는 2%p입니다.

전체 품목의 날짜별 기록을 평가한 결과이며, 개별 제품의 불량을 미리 판정하는 기능은 아닙니다.


In [ ]:
model_features=['density_set_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s','dose_etch_chromic_automatic_ml','dose_chrome_chromic_automatic_ml','dose_additive_automatic_ml']
common_process=np.logical_and.reduce([lag_matrices[k][['density_set_a_dm2','temperature_c_avg','duration_actual_s']].notna().all(axis=1).to_numpy() for k in LAGS])
folds=pd.read_csv(ROOT/'prepared/five_fold/fold_assignments.csv',parse_dates=['date'])
print(f'검사 기록 {len(valid)}일 중 작업 기록을 함께 비교할 수 있는 {int(common_process.sum())}일로 예측을 평가합니다.')
def new_model(name):
    estimator=Ridge(alpha=10) if name=='Ridge' else RandomForestRegressor(n_estimators=160,max_depth=4,min_samples_leaf=5,random_state=SEED,n_jobs=1)
    return make_pipeline(SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True),StandardScaler(),estimator)
def metrics(y,pred,w):
    return {'mae_pp':mean_absolute_error(y,pred),'rmse_pp':float(np.sqrt(mean_squared_error(y,pred))),'weighted_mae_pp':float(np.average(np.abs(np.asarray(y)-pred),weights=w)),'r2':r2_score(y,pred) if len(y)>1 else np.nan}
cv_rows,cv_predictions=[],[]
for lag in LAGS:
    X=lag_matrices[lag][model_features]
    for fold in sorted(pd.to_numeric(folds.fold_id,errors='coerce').dropna().unique()):
        fa=folds[folds.fold_id.astype(str).eq(str(int(fold)))]
        tr=valid.index[valid.date.isin(fa[fa.role.eq('train')].date)&common_process]
        va=valid.index[valid.date.isin(fa[fa.role.eq('validation')].date)&common_process]
        if len(tr)<15 or len(va)<3:
            cv_rows.append({'lag_days':lag,'fold':int(fold),'model':'skipped','train_days':len(tr),'validation_days':len(va)});continue
        y=valid.loc[tr,'pinhole_pit_pphv'];yv=valid.loc[va,'pinhole_pit_pphv']
        for name in ['Median baseline','Ridge','Random Forest']:
            if name=='Median baseline':pred=np.repeat(y.median(),len(va))
            else:
                fitted=new_model(name);fitted.fit(X.loc[tr],y);pred=np.clip(fitted.predict(X.loc[va]),0,None)
            cv_rows.append({'lag_days':lag,'fold':int(fold),'model':name,'train_days':len(tr),'validation_days':len(va),**metrics(yv,pred,valid.loc[va,'total_reviewed'])})
            cv_predictions.extend({'lag_days':lag,'fold':int(fold),'model':name,'date':valid.loc[idx,'date'],'actual':float(actual),'predicted':float(pr),'reviewed':float(valid.loc[idx,'total_reviewed'])} for idx,actual,pr in zip(va,yv,pred))
cv=pd.DataFrame(cv_rows);cvp=pd.DataFrame(cv_predictions)
save(cv,'temporal_cv_folds');save(cvp,'temporal_cv_predictions')
cv_summary=[]
for (lag,name),g in cvp.groupby(['lag_days','model']):cv_summary.append({'lag_days':lag,'model':name,'validation_days':len(g),'fold_count':g.fold.nunique(),**metrics(g.actual,g.predicted.to_numpy(),g.reviewed)})
cv_summary=pd.DataFrame(cv_summary).sort_values('mae_pp');save(cv_summary,'temporal_cv_summary');show(cv_summary)
chosen=cv_summary[cv_summary.model.ne('Median baseline')].iloc[0]
chosen_lag=int(chosen.lag_days);chosen_name=chosen.model
tr=valid.index[(valid.date<pd.Timestamp('2026-05-01'))&common_process]
te=valid.index[(valid.date>=pd.Timestamp('2026-05-01'))&common_process]
fitted=new_model(chosen_name);fitted.fit(lag_matrices[chosen_lag].loc[tr,model_features],valid.loc[tr,'pinhole_pit_pphv'])
test_pred=np.clip(fitted.predict(lag_matrices[chosen_lag].loc[te,model_features]),0,None)
baseline_pred=np.repeat(valid.loc[tr,'pinhole_pit_pphv'].median(),len(te))
holdout=pd.DataFrame([{'model':name,'lag_days':chosen_lag,'train_days':len(tr),'test_days':len(te),**metrics(valid.loc[te,'pinhole_pit_pphv'],pred,valid.loc[te,'total_reviewed'])} for name,pred in [(chosen_name,test_pred),('Median baseline',baseline_pred)]])
save(holdout,'holdout_may_metrics');show(holdout)
test_predictions=valid.loc[te,['date','pinhole_pit_pphv','total_reviewed']].copy();test_predictions['prediction']=test_pred;test_predictions['baseline']=baseline_pred
save(test_predictions,'holdout_may_predictions')

## 14. 그래프 보기

월별 검사 결과, 작업조건, 검사일과 작업일의 관계, 예측 결과를 그래프로 확인합니다.


In [ ]:
fig,ax=plt.subplots(figsize=(10.5,4.4))
x=np.arange(len(monthly));ax.plot(x,monthly.total_pphv,'o-',label='전체 불량 PPHV',color='#355C7D');ax.plot(x,monthly.pinhole_pit_pphv,'o-',label='핀홀·피트 PPHV',color='#B6563B')
ax.set_xticks(x,monthly.month,rotation=25);ax.set_ylabel('불량건수 / 검사수량 × 100');ax.set_title('월별 출하검사 결과');ax.legend(frameon=False);ax.grid(axis='y',alpha=.2)
fig.tight_layout();fig.savefig(FIGURES/'01_monthly_defects.png');plt.close(fig)
fig,axes=plt.subplots(4,1,figsize=(11,9.5),sharex=True)
for ax,col,label in zip(axes,['density_set_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s'],['설정 전류밀도 (A/dm²)','전압 (V, >0.1)','평균 온도 (℃)','실제 처리시간 (초)']):
    for item,g in item_daily.groupby('item'):ax.plot(g.date,g[col],'.',ms=3,label=item,alpha=.8)
    ax.set_ylabel(label);ax.grid(axis='y',alpha=.2)
axes[0].legend(frameon=False,fontsize=8,ncol=3);axes[0].set_title('대표품목 크롬 공정의 일평균 조건');fig.tight_layout();fig.savefig(FIGURES/'02_representative_conditions.png');plt.close(fig)
heat_features=['density_set_a_dm2','voltage_clean_v','temperature_c_avg','duration_actual_s','dose_etch_chromic_net_ml','dose_chrome_chromic_net_ml','dose_additive_net_ml','도금조_L1_이물질','도금조_L3_이물질','도금조_L4_이물질']
heat=correlations[(correlations.target=='pinhole_pit_pphv')&(correlations['sample']=='common_3_lags')&correlations.feature.isin(heat_features)].pivot(index='feature',columns='lag_days',values='spearman_r').reindex(heat_features)
heatn=correlations[(correlations.target=='pinhole_pit_pphv')&(correlations['sample']=='common_3_lags')&correlations.feature.isin(heat_features)].pivot(index='feature',columns='lag_days',values='n_days').reindex(heat_features)
fig,ax=plt.subplots(figsize=(9.5,6.8));im=ax.imshow(heat.values,cmap='RdBu_r',vmin=-1,vmax=1,aspect='auto')
labels=['설정 전류밀도','전압 (센서 진단 제외)','평균 온도','실제 처리시간','에칭 크롬산 순투입(ml)','도금 크롬산 순투입(ml)','첨가제 순투입(ml)','도금액 1라인 이물질','도금액 3라인 이물질','도금액 4라인 이물질']
ax.set_yticks(range(len(heat)),labels);ax.set_xticks(range(3),[f'{lag}일' for lag in LAGS])
for i in range(len(heat)):
    for j in range(3):
        v=heat.iloc[i,j];ax.text(j,i,f'{v:.2f}\nn={heatn.iloc[i,j]:.0f}' if pd.notna(v) else f'표본 부족\nn={heatn.iloc[i,j]:.0f}',ha='center',va='center',color='white' if abs(v)>.65 else '#17232D',fontsize=9)
ax.set_title('전체품목 핀홀·피트 PPHV와 시차별 순위상관\n성분 라인과 생산설비의 대응 미확정');fig.colorbar(im,ax=ax,label='Spearman r',shrink=.75);fig.tight_layout();fig.savefig(FIGURES/'03_lag_comparison.png');plt.close(fig)
sol_features=[f'도금조_L{line}_{comp}' for line in [1,3,4] for comp in ['크롬산','3가크롬','철분','황산','이물질']]
sa=correlations[(correlations.target=='pinhole_pit_pphv')&(correlations['sample']=='available')&correlations.feature.isin(sol_features)]
sh=sa.pivot(index='feature',columns='lag_days',values='spearman_r').reindex(sol_features)
sn=sa.pivot(index='feature',columns='lag_days',values='n_days').reindex(sol_features)
fig,ax=plt.subplots(figsize=(9,8));im=ax.imshow(sh.values,cmap='RdBu_r',vmin=-1,vmax=1,aspect='auto')
ax.set_yticks(range(len(sh)),[f'{line}라인 {comp}' for line in [1,3,4] for comp in ['크롬산','3가크롬','철분','황산','이물질']]);ax.set_xticks(range(3),[f'{k}일' for k in LAGS])
for i in range(len(sh)):
    for j in range(3):
        v=sh.iloc[i,j];ax.text(j,i,f'{v:.2f} (n={sn.iloc[i,j]:.0f})' if pd.notna(v) else f'n={sn.iloc[i,j]:.0f}',ha='center',va='center',color='white' if abs(v)>.65 else '#17232D',fontsize=9)
ax.set_title('도금액 성분과 전체품목 핀홀·피트 PPHV\n각 시차별 가용 표본. 동일 표본 비교 아님');fig.colorbar(im,ax=ax,label='Spearman r',shrink=.7);fig.tight_layout();fig.savefig(FIGURES/'05_solution_available_lags.png');plt.close(fig)
fig,ax=plt.subplots(figsize=(10.5,4.3));ax.plot(test_predictions.date,test_predictions.pinhole_pit_pphv,'o-',label='실측');ax.plot(test_predictions.date,test_predictions.prediction,'o-',label=f'{chosen_name}, {chosen_lag}일');ax.plot(test_predictions.date,test_predictions.baseline,'--',label='학습 중앙값 기준선')
ax.set_ylabel('핀홀·피트 PPHV');ax.set_title('2026-05 최종 시험: 전체품목 일별 집계');ax.legend(frameon=False);ax.grid(axis='y',alpha=.2);fig.autofmt_xdate();fig.tight_layout();fig.savefig(FIGURES/'04_holdout_predictions.png');plt.close(fig)
for fname in ['01_monthly_defects.png','03_lag_comparison.png']:display(Image(filename=str(FIGURES/fname)))

## 15. 계산 결과 점검

계산한 합계가 원본과 맞는지, 중복된 날짜는 없는지, 입력파일이 바뀌지 않았는지 확인합니다.
아래 표의 **통과**가 모두 **True**이면 이 점검을 통과한 것입니다.


In [ ]:
import xlrd
extra_checks=[]
for _,r in manifest[manifest.source_group.isin(['chemical_inputs','power_usage','production_history'])].iterrows():
    source=ROOT/r.source_relative_path;checked=0;errors=0
    if r.source_group=='production_history':
        with (ROOT/r.csv_relative_path).open(encoding='utf-8-sig',newline='') as f:
            reader=csv.reader(f);ar=[next(reader) for _ in range(4)]
        wb=load_workbook(source,read_only=True,data_only=True);ws=wb[r.table_name];rr=list(ws.iter_rows(min_row=1,max_row=4,values_only=True))
        for i in range(1,4):
            for j in [0,1,2,3,7,9,11,13,17,19]:
                value=rr[i][j];csvvalue=ar[i][j];checked+=1
                if isinstance(value,datetime):same=pd.Timestamp(value)==pd.Timestamp(csvvalue)
                elif isinstance(value,(int,float)):same=np.isclose(value,num(csvvalue),equal_nan=True)
                else:same=str(value or '').replace('\r\n','\n')==csvvalue.replace('\r\n','\n')
                errors+=not same
        wb.close()
    else:
        ar=rows_for(ROOT/r.csv_relative_path);wb=xlrd.open_workbook(source,on_demand=True);ws=wb.sheet_by_name(r.table_name)
        numeric_cells=[(i,j) for i,row in enumerate(ar) for j,v in enumerate(row) if pd.notna(num(v))]
        # 원본의 처음·중간·마지막에서 일부 숫자를 골라 계산표와 비교합니다.
        pick=sorted(set(np.linspace(0,len(numeric_cells)-1,min(90,len(numeric_cells)),dtype=int)))
        for k in pick:
            i,j=numeric_cells[k];value=ws.cell_value(i,j);checked+=1
            errors+=not np.isclose(num(value),num(ar[i][j]),equal_nan=True)
        wb.release_resources()
    extra_checks.append({'source_file':r.source_relative_path,'cells_checked':checked,'mismatches':errors})
all_raw_checks=pd.concat([pd.DataFrame(raw_checks),pd.DataFrame(extra_checks)],ignore_index=True);save(all_raw_checks,'original_cell_checks')
solution_checks=[]
for (source,sheet),g in solution_all.groupby(['source_file','source_sheet']):
    selected=g[g.value_raw.notna()].iloc[np.linspace(0,g.value_raw.notna().sum()-1,min(100,g.value_raw.notna().sum()),dtype=int)]
    wb=load_workbook(ROOT/source,read_only=True,data_only=True);ws=wb[sheet]
    rr=list(ws.iter_rows(min_row=1,max_row=int(selected.source_row.max()),values_only=True));errors=0
    for _,row in selected.iterrows():errors+=not np.isclose(num(rr[int(row.source_row)-1][int(row.source_col)-1]),row.value_raw,equal_nan=True)
    solution_checks.append({'source_file':source,'cells_checked':len(selected),'mismatches':errors});wb.close()
all_raw_checks=pd.concat([all_raw_checks,pd.DataFrame(solution_checks)],ignore_index=True);save(all_raw_checks,'original_cell_checks')
verification=[
 ['검사 날짜 중복 없음',not daily.date.duplicated().any(),f'{len(daily)} 날짜'],
 ['206일 표본 재현',len(valid)==206,'검사수량>0'],
 ['유효 206일 기존 불량·검사수량 일치',bool((repro.loc[repro.valid_denominator,['count_difference','reviewed_difference']]==0).all().all()),'원본 유효 분모 날짜만 비교'],
 ['기존 0 대체와 원본 빈칸 구분',int(repro.prepared_zero_for_source_blank.sum())==6,'검사수량 빈칸6일을 결측 유지'],
 ['불량유형 합계=Total',bool(np.allclose(daily.total_defects,daily.defect_type_sum,equal_nan=False)),'26개 유형'],
 ['원본 대조',bool(all_raw_checks.mismatches.eq(0).all()),f'{int(all_raw_checks.cells_checked.sum())} 셀; {all_raw_checks.source_file.nunique()} 파일'],
 ['약품 날짜·종류 중복 없음',not chem.duplicated(['date','chemical']).any(),f'{len(chem)} 일자×약품'],
 ['약품 월계=일계(0.1ml 허용)',bool(chem_audit.monthly_minus_daily.abs().max()<.1),f'최대차 {chem_audit.monthly_minus_daily.abs().max():.6f}ml'],
 ['생산 공정행 중복 제거',not chrome.duplicated(dupekeys).any(),f'제거 {int(duplicate_mask.sum())}행'],
 ['상위 사본과 raw 동일',bool(pd.DataFrame(duplicates).hash_equal.all()),f'{len(duplicates)} 사본'],
 ['시차 방향',all((valid.date-(valid.date-pd.to_timedelta(k,unit='D'))).dt.days.eq(k).all() for k in LAGS),'검사일 t, 공정일 t-lag'],
 ['최종 시험 분리',valid.loc[tr,'date'].max()<valid.loc[te,'date'].min(),'5월 시험, 이전월 학습'],
 ['공통 표본 시간순 5개 검증',cvp.fold.nunique()==5,f'{cvp.fold.nunique()}개'],
]
unchanged_original=all(digest(ROOT/path)==h for path,h in initial_hashes.items())
unchanged_intermediate=all(digest(ROOT/path)==h for path,h in input_fingerprints.items())
verification += [['원본 해시 보존',unchanged_original,f'{len(initial_hashes)} 파일'],['기존 CSV/prepared/지시문 해시 보존',unchanged_intermediate,f'{len(input_fingerprints)} 파일']]
verification=pd.DataFrame(verification,columns=['check','passed','detail']);save(verification,'verification')
show(verification)
assert verification.passed.all(), '검증 실패는 verification.csv에서 확인하세요. 결과 해석 전에 해소해야 합니다.'

## 16. 새 자료 반영

`추가자료/입력`에 넣은 양식 파일을 읽어 기존 자료와 함께 분석합니다. 하루의 전체 품목 집계를 한 행으로 입력합니다.
검사수량과 핀홀피트건수가 있는 행은 학습에 쓰고, 두 칸이 비어 있는 행은 작업조건으로 예측만 합니다.
추가자료가 없으면 기존 분석 결과만 저장합니다. 양식과 입력 폴더는 첫 실행 때 자동으로 만들어집니다.


In [ ]:
# 정해진 양식의 추가자료를 읽고, 기존 자료와 함께 모델을 다시 학습합니다.
ADDITIONAL_COLUMNS = {
    '날짜': 'date',
    '검사수량': 'total_reviewed',
    '핀홀피트건수': 'pinhole_pit_count',
    '설정전류밀도_A_dm2': 'density_set_a_dm2',
    '전압_V': 'voltage_clean_v',
    '온도_C': 'temperature_c_avg',
    '처리시간_초': 'duration_actual_s',
    '에칭크롬산자동투입_ml': 'dose_etch_chromic_automatic_ml',
    '도금크롬산자동투입_ml': 'dose_chrome_chromic_automatic_ml',
    '첨가제자동투입_ml': 'dose_additive_automatic_ml',
}
ADDITIONAL_FEATURES = list(ADDITIONAL_COLUMNS.values())[3:]


def prepare_additional_folders(root):
    from pathlib import Path
    import pandas as pd

    root = Path(root)
    inbox = root / '추가자료' / '입력'
    templates = root / '추가자료' / '양식'
    for directory in [inbox, templates]:
        directory.mkdir(parents=True, exist_ok=True)
    blank = pd.DataFrame(columns=list(ADDITIONAL_COLUMNS))
    csv_template = templates / '추가자료_입력양식.csv'
    excel_template = templates / '추가자료_입력양식.xlsx'
    if not csv_template.exists():
        blank.to_csv(csv_template, index=False, encoding='utf-8-sig')
    if not excel_template.exists():
        blank.to_excel(excel_template, index=False, sheet_name='입력자료')
    return inbox


def read_additional_files(inbox, last_historical_date):
    from datetime import date, datetime
    import hashlib
    import io
    import re
    import numpy as np
    import pandas as pd

    frames, source_files = [], []
    paths = sorted(p for p in inbox.iterdir()
                   if p.is_file() and not p.name.startswith(('.', '~$')))
    for path in paths:
        if path.suffix.lower() not in {'.csv', '.xlsx'}:
            raise ValueError(f'{path.name}: CSV 또는 정해진 양식의 Excel(.xlsx) 파일을 넣어 주세요.')
        payload = path.read_bytes()
        if path.suffix.lower() == '.csv':
            try:
                text = payload.decode('utf-8-sig')
            except UnicodeDecodeError:
                text = payload.decode('cp949')
            table = pd.read_csv(io.StringIO(text), dtype=object, keep_default_na=False)
        else:
            table = pd.read_excel(io.BytesIO(payload), sheet_name=0, dtype=object,
                                  keep_default_na=False, engine='openpyxl')
        table.columns = [str(c).strip() for c in table.columns]
        required = list(ADDITIONAL_COLUMNS)[:7]
        missing = [c for c in required if c not in table.columns]
        unknown = [c for c in table.columns if c not in ADDITIONAL_COLUMNS]
        if missing or unknown:
            details = []
            if missing:
                details.append('없는 열: ' + ', '.join(missing))
            if unknown:
                details.append('양식에 없는 열: ' + ', '.join(unknown))
            raise ValueError(f'{path.name}: ' + '; '.join(details) + '. 제공된 입력양식을 사용해 주세요.')
        blank = table.apply(lambda col: col.astype(str).str.strip().eq('')).all(axis=1)
        table = table.loc[~blank].copy()
        table['_source_row'] = table.index + 2
        if table.empty:
            raise ValueError(f'{path.name}: 입력된 자료가 없습니다. 빈 양식은 입력 폴더에서 빼 주세요.')
        for column in ADDITIONAL_COLUMNS:
            if column not in table:
                table[column] = ''

        def fail(mask, message):
            if mask.any():
                rows = ', '.join(map(str, table.loc[mask, '_source_row'].head(5)))
                raise ValueError(f'{path.name} {rows}행: {message}')

        dates = []
        for value in table['날짜']:
            if isinstance(value, (pd.Timestamp, datetime, date)):
                timestamp = pd.Timestamp(value)
                dates.append(timestamp.strftime('%Y-%m-%d') if timestamp == timestamp.normalize() else '')
            else:
                text = str(value).strip()
                dates.append(text if re.fullmatch(r'\d{4}-\d{2}-\d{2}', text) else '')
        parsed_dates = pd.Series(pd.to_datetime(dates, format='%Y-%m-%d', errors='coerce'),
                                 index=table.index)
        fail(parsed_dates.isna(), '날짜를 YYYY-MM-DD 형식으로 입력해 주세요.')
        fail(parsed_dates.le(pd.Timestamp(last_historical_date)),
             f'기존 검사자료의 마지막 날짜({pd.Timestamp(last_historical_date):%Y-%m-%d}) 이후 자료를 넣어 주세요.')
        table['날짜'] = parsed_dates
        for column in list(ADDITIONAL_COLUMNS)[1:]:
            raw = table[column].astype(str).str.strip().str.replace(',', '', regex=False)
            numbers = pd.to_numeric(raw.replace('', np.nan), errors='coerce')
            fail(raw.ne('') & (~np.isfinite(numbers)), f'{column}에는 숫자를 입력해 주세요.')
            fail(numbers.lt(0), f'{column}에는 0 이상의 값을 입력해 주세요.')
            table[column] = numbers
        for column in required[3:]:
            fail(table[column].isna() | table[column].le(0), f'{column}에는 0보다 큰 값을 입력해 주세요.')
        fail(table['전압_V'].le(0.1), '전압_V에는 0.1V 이하 기록을 제외한 크롬 공정 일평균을 입력해 주세요.')
        inspected, defects = table['검사수량'], table['핀홀피트건수']
        fail(inspected.isna() ^ defects.isna(), '검사수량과 핀홀피트건수는 함께 입력하거나 둘 다 비워 주세요.')
        for column in ['검사수량', '핀홀피트건수']:
            fail(table[column].notna() & table[column].mod(1).ne(0), f'{column}에는 정수를 입력해 주세요.')
        fail(inspected.eq(0) & defects.gt(0), '검사수량이 0이면 불량 건수도 0이어야 합니다.')
        table['_source_file'] = path.name
        frames.append(table.rename(columns=ADDITIONAL_COLUMNS))
        source_files.append({'file': path.name, 'sha256': hashlib.sha256(payload).hexdigest(),
                             'rows': len(table)})
    if not frames:
        return pd.DataFrame(), source_files
    result = pd.concat(frames, ignore_index=True).sort_values('date').reset_index(drop=True)
    duplicates = result.duplicated('date', keep=False)
    if duplicates.any():
        details = ', '.join(f'{r.date:%Y-%m-%d} ({r["_source_file"]})'
                            for _, r in result.loc[duplicates].head(5).iterrows())
        raise ValueError('같은 날짜의 자료가 중복되었습니다. 하루에 전체 품목 합계 한 행만 넣어 주세요: ' + details)
    return result, source_files


def analyze_additional_data(root, output, historical, historical_features, seed=20261012):
    from pathlib import Path
    import hashlib
    import html
    import json
    import math
    import os
    from datetime import datetime, timezone
    import joblib
    import numpy as np
    import pandas as pd
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import Ridge
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    from sklearn.model_selection import TimeSeriesSplit
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    root, output = Path(root), Path(output)
    inbox = prepare_additional_folders(root)
    destination = output / '추가자료분석'
    destination.mkdir(parents=True, exist_ok=True)
    status_file = destination / '실행정보.json'
    info = {'status': 'running', 'started_at': datetime.now(timezone.utc).isoformat()}

    def write_status():
        status_file.write_text(json.dumps(info, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

    def store_table(frame, name):
        frame.to_csv(destination / name, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')

    def model(name):
        estimator = Ridge(alpha=10) if name == 'Ridge' else RandomForestRegressor(
            n_estimators=160, max_depth=4, min_samples_leaf=5, random_state=seed, n_jobs=1)
        return make_pipeline(SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True),
                             StandardScaler(), estimator)

    def scores(actual, predicted):
        return {'평균 예측 오차(%p)': float(mean_absolute_error(actual, predicted)),
                '큰 오차를 반영한 값(%p)': float(np.sqrt(mean_squared_error(actual, predicted)))}

    write_status()
    try:
        extra, sources = read_additional_files(inbox, historical['date'].max())
        info.update({'input_files': sources, 'input_rows': len(extra)})
        if extra.empty:
            info['status'] = 'no_input'
            write_status()
            return {'input_rows': 0, 'message': '추가자료가 없습니다. 기존 자료의 분석 결과를 저장합니다. 새 자료는 왼쪽 `추가자료/양식`의 양식을 작성해 `추가자료/입력`에 올린 뒤 전체 실행해 주세요.',
                    'summary_html': '', 'info': info}

        base = historical[['date', 'total_reviewed', 'pinhole_pit_count']].copy().reset_index(drop=True)
        base['date'] = pd.to_datetime(base['date']).dt.normalize()
        history_x = historical_features.reindex(pd.DatetimeIndex(base['date']))[ADDITIONAL_FEATURES]
        base = pd.concat([base, history_x.reset_index(drop=True)], axis=1)
        base['_source_file'] = '기존 검증자료'
        joined = pd.concat([base, extra], ignore_index=True).sort_values('date').reset_index(drop=True)
        if joined['date'].duplicated().any():
            raise ValueError('기존 자료와 추가자료에 같은 날짜가 있습니다. 담당자에게 확인을 요청해 주세요.')
        joined['target'] = joined['pinhole_pit_count'] / joined['total_reviewed'].where(joined['total_reviewed'].gt(0)) * 100
        observed = joined.loc[joined['target'].notna()].copy()
        observed['월'] = observed['date'].dt.strftime('%Y-%m')
        monthly = observed.groupby('월', as_index=False).agg(
            검사일수=('date', 'size'), 검사수량=('total_reviewed', 'sum'),
            핀홀피트건수=('pinhole_pit_count', 'sum'))
        monthly['핀홀피트기록 비율(%)'] = monthly['핀홀피트건수'] / monthly['검사수량'] * 100
        core = ['density_set_a_dm2', 'temperature_c_avg', 'duration_actual_s']
        usable = joined[core].notna().all(axis=1) & joined[core].gt(0).all(axis=1)
        labeled = joined.loc[joined['target'].notna() & usable].copy().reset_index(drop=True)
        new_labeled = extra['total_reviewed'].gt(0)
        info.update({'historical_rows': len(base), 'training_rows': len(labeled),
                     'new_labeled_rows': int(new_labeled.sum()),
                     'prediction_only_rows': int((~new_labeled).sum()),
                     'target': '핀홀피트건수 / 검사수량 * 100',
                     'feature_columns': ADDITIONAL_FEATURES,
                     'data_unit': '전체 품목의 하루 집계; 작업조건은 크롬 공정의 일평균, 약품은 하루 자동투입 합계'})
        if len(labeled) < 60:
            raise ValueError(f'작업조건과 검사 결과가 함께 있는 날짜가 {len(labeled)}일입니다. 모델을 학습하려면 60일 이상이 필요합니다.')
        pending = extra.loc[~new_labeled].copy()
        last_label = labeled['date'].max()
        if not pending.empty and pending['date'].le(last_label).any():
            raise ValueError(f'검사 결과가 없는 예측용 자료는 마지막 학습일({last_label:%Y-%m-%d}) 이후 날짜로 넣어 주세요.')
        test_count = max(10, math.ceil(len(labeled) * 0.2))
        training, testing = labeled.iloc[:-test_count], labeled.iloc[-test_count:]
        cv_rows = []
        for fold, (train_idx, check_idx) in enumerate(TimeSeriesSplit(n_splits=3).split(training), 1):
            for name in ['Ridge', 'Random Forest']:
                candidate = model(name)
                candidate.fit(training.iloc[train_idx][ADDITIONAL_FEATURES], training.iloc[train_idx]['target'])
                predicted = np.clip(candidate.predict(training.iloc[check_idx][ADDITIONAL_FEATURES]), 0, None)
                cv_rows.append({'방법': name, '검증 구간': fold, '학습 날짜 수': len(train_idx),
                                '검증 날짜 수': len(check_idx),
                                **scores(training.iloc[check_idx]['target'], predicted)})
        cv = pd.DataFrame(cv_rows)
        ranking = cv.groupby('방법', as_index=False)['평균 예측 오차(%p)'].mean().sort_values(
            ['평균 예측 오차(%p)', '방법'], kind='stable')
        selected = str(ranking.iloc[0]['방법'])
        evaluation_model = model(selected)
        evaluation_model.fit(training[ADDITIONAL_FEATURES], training['target'])
        predictions = np.clip(evaluation_model.predict(testing[ADDITIONAL_FEATURES]), 0, None)
        baseline = np.repeat(training['target'].median(), len(testing))
        evaluation = pd.DataFrame([
            {'방법': selected, '학습 날짜 수': len(training), '시험 날짜 수': len(testing),
             **scores(testing['target'], predictions)},
            {'방법': '과거 중간값을 그대로 예측', '학습 날짜 수': len(training), '시험 날짜 수': len(testing),
             **scores(testing['target'], baseline)},
        ])
        test_output = pd.DataFrame({'날짜': testing['date'], '실제 핀홀피트기록 비율(%)': testing['target'],
                                    '예측 핀홀피트기록 비율(%)': predictions,
                                    '과거 중간값 예측(%)': baseline})
        final_model = model(selected)
        final_model.fit(labeled[ADDITIONAL_FEATURES], labeled['target'])
        pending_output = pd.DataFrame(columns=['날짜', '예측 핀홀피트기록 비율(%)'])
        if not pending.empty:
            pending_output = pd.DataFrame({'날짜': pending['date'].to_numpy(),
                '예측 핀홀피트기록 비율(%)': np.clip(final_model.predict(pending[ADDITIONAL_FEATURES]), 0, None)})
        if not np.isfinite(predictions).all() or not np.isfinite(pending_output.iloc[:, 1].astype(float)).all():
            raise ValueError('예측값을 계산하지 못했습니다. 입력값과 단위를 담당자에게 확인해 주세요.')
        for source in sources:
            if hashlib.sha256((inbox / source['file']).read_bytes()).hexdigest() != source['sha256']:
                raise ValueError('실행 중 추가자료가 바뀌었습니다. 파일 업로드를 마친 뒤 다시 실행해 주세요.')

        store_table(cv, '시간순_모델비교.csv')
        store_table(monthly, '월별_검사집계.csv')
        store_table(evaluation, '검증결과.csv')
        store_table(test_output, '검증용_예측결과.csv')
        store_table(pending_output, '미검사자료_예측결과.csv')
        store_table(labeled[['date', 'total_reviewed', 'pinhole_pit_count', 'target', *ADDITIONAL_FEATURES]],
                    '학습에_사용한_일별자료.csv')
        info.update({'selected_model': selected,
                     'training_start': str(labeled['date'].min().date()),
                     'training_end': str(last_label.date()),
                     'evaluation_training_end': str(training['date'].max().date()),
                     'evaluation_test_start': str(testing['date'].min().date()),
                     'evaluation_test_end': str(testing['date'].max().date()),
                     'selection': '학습 구간에서 시간순 3회 비교; 마지막 20%는 별도 시험',
                     'model_file': '핀홀피트_예측모델.joblib',
                     'model_refit': '저장 모델은 시험을 마친 뒤 검사 결과가 있는 모든 학습 가능 날짜로 다시 학습'})
        temporary_model = destination / '.model-writing.joblib'
        joblib.dump({'model': final_model, 'feature_columns': ADDITIONAL_FEATURES,
                     'metadata': {k: v for k, v in info.items() if k != 'status'}},
                    temporary_model)
        os.replace(temporary_model, destination / info['model_file'])
        file_names = ', '.join(html.escape(s['file']) for s in sources)
        labeled_note = (f'검사 결과가 있는 추가자료 {int(new_labeled.sum())}일을 학습에 포함했습니다.'
                        if new_labeled.any() else '추가자료에는 검사 결과가 없어 기존 검사자료로 학습하고 새 작업조건의 예측만 계산했습니다.')
        summary = (f'<p>추가자료 {len(sources)}개 파일, {len(extra)}일을 읽었습니다. {labeled_note}</p>'
                   f'<p>현재 모델: {html.escape(selected)} · 학습에 사용한 날짜: {len(labeled)}일</p>'
                   '<p>마지막 20% 날짜를 따로 남겨 실제 결과와 비교했습니다. 평균 예측 오차는 작을수록 좋습니다. '
                   '저장된 모델은 이 시험을 마친 뒤 검사 결과가 있는 모든 날짜로 다시 학습했습니다.</p>'
                   + evaluation.to_html(index=False, escape=True, border=0, float_format=lambda x: f'{x:,.3f}')
                   + '<p>작업조건이 입력된 날짜의 핀홀·피트 기록 비율을 예측합니다. 개별 제품의 합격 판정이나 생산 전 경보 기능은 아닙니다.</p>')
        body = (summary + '<h2>기존 자료와 추가자료를 합한 월별 검사 결과</h2>'
                + monthly.to_html(index=False, escape=True, border=0, float_format=lambda x: f'{x:,.3f}')
                + '<h2>검사 결과가 없는 추가자료의 예측</h2>'
                + (pending_output.to_html(index=False, escape=True, border=0) if len(pending_output)
                   else '<p>검사 결과가 없는 추가자료는 없습니다.</p>')
                + f'<h2>입력 파일</h2><p>{file_names}</p>'
                + '<p>원본·기존 입력자료는 변경하지 않았습니다. 입력 파일 확인값과 학습 기간은 실행정보.json에 저장했습니다.</p>')
        report = ('<!doctype html><html lang="ko"><meta charset="UTF-8">'
                  '<meta name="viewport" content="width=device-width,initial-scale=1">'
                  '<title>한국로텍 추가자료 분석</title><style>'
                  'body{font-family:Arial,sans-serif;max-width:1080px;margin:32px auto;padding:24px;line-height:1.7}'
                  'table{border-collapse:collapse;width:100%}td,th{padding:10px;border-bottom:1px solid #ddd;text-align:left}'
                  '</style><h1>한국로텍 추가자료 분석</h1>' + body + '</html>')
        (destination / '추가자료_분석보고서.html').write_text(report, encoding='utf-8')
        info['status'] = 'passed'
        write_status()
        return {'input_rows': len(extra), 'info': info,
                'message': f'추가자료 {len(extra)}일을 반영했습니다. {labeled_note} [추가자료 보고서 열기](분석결과/추가자료분석/추가자료_분석보고서.html)',
                'summary_html': summary + '<p><a href="추가자료분석/추가자료_분석보고서.html">추가자료 상세 보고서 열기</a></p>'}
    except Exception as error:
        info.update({'status': 'failed', 'error': str(error)})
        write_status()
        raise


additional_result = analyze_additional_data(ROOT, OUT, valid, features, seed=SEED)
display(Markdown(additional_result['message']))


## 17. 보고서 저장

분석 보고서와 계산표·그래프를 저장합니다. 실행이 끝나면 아래에 **분석 완료**와 보고서 링크가 표시됩니다.


In [ ]:
total_defects = float(valid.total_defects.sum())
reviewed_total = float(valid.total_reviewed.sum())
pinhole_total = float(valid.pinhole_pit_count.sum())
headline = {
    'inspection_calendar_days': len(daily),
    'inspection_valid_days': len(valid),
    'first_inspection_date': str(valid.date.min().date()),
    'last_inspection_date': str(valid.date.max().date()),
    'total_defects': total_defects,
    'total_reviewed': reviewed_total,
    'weighted_total_pphv': 100 * total_defects / reviewed_total,
    'pinhole_pit_count': pinhole_total,
    'weighted_pinhole_pit_pphv': 100 * pinhole_total / reviewed_total,
    'pinhole_pit_share_of_defects': pinhole_total / total_defects,
    'production_raw_rows': int(pd.DataFrame(production_audit).data_rows.sum()),
    'production_station_present_rows': int(pd.DataFrame(production_audit).station_present_rows.sum()),
    'production_entirely_blank_rows': int(pd.DataFrame(production_audit).entirely_blank_rows.sum()),
    'production_station_missing_nonempty_rows': int(pd.DataFrame(production_audit).station_missing_nonempty_rows.sum()),
    'representative_four_conditions_present': 12902,
    'representative_four_conditions_valid': int(joins.loc[joins['join'].eq('대표품목 공정4조건 유효'), 'matched'].iloc[0]),
    'thickness_records_other_parts': len(thickness_records),
    'representative_thickness_records': int(thickness_records.is_representative.sum()),
    'unique_image_assets': len(image_tables['이미지별_분류']),
    'chrome_events': len(chrome),
    'active_chrome_events': len(active_chrome),
    'representative_events': len(representative),
    'strict_lot_linked_records': 0,
    'model_common_days': int(common_process.sum()),
    'chosen_lag': chosen_lag,
    'chosen_model': chosen_name,
    'original_files': len(inventory),
    'source_conflict_keys': len(conflict_keys),
}
(OUT / 'summary.json').write_text(json.dumps(headline, ensure_ascii=False, indent=2), encoding='utf-8')

coverage=pd.DataFrame([
 ['생산 변환 범위',14,int(pd.DataFrame(production_audit).data_rows.sum()),str(pd.DataFrame(production_audit).first_valid_end_time.min().date()),str(pd.DataFrame(production_audit).last_valid_end_time.max().date()),'설비명 없는 행·빈 서식행 포함. 유효 공정 또는 독립 제품 수가 아님'],
 ['설비명이 있는 생산기록',14,int(pd.DataFrame(production_audit).station_present_rows.sum()),str(chrome.date.min().date()),str(chrome.date.max().date()),'설비명이 있는 기록행. 기간은 크롬공정 관측 기준. 재통과 포함 가능'],
 ['크롬 공정',14,len(chrome),str(chrome.date.min().date()),str(chrome.date.max().date()),'0전류/면적/처리시간은 활성 분석에서 제외'],
 ['대표품목 크롬 공정',14,len(representative),str(representative.date.min().date()),str(representative.date.max().date()),'3개 정확한 품번'],
 ['출하검사 일자',7,len(daily),str(daily.date.min().date()),str(daily.date.max().date()),f'유효 분모 {len(valid)}일'],
 ['약품 일계',42,len(chem),str(chem.date.min().date()),str(chem.date.max().date()),'일자×약품3종'],
 ['액 성분 유효 측정',2,int(solution.value.notna().sum()),str(solution[solution.value.notna()].date.min().date()),str(solution[solution.value.notna()].date.max().date()),'일자×라인×액조×성분'],
 ['기타품목 두께 기록',1,len(thickness_records),'2026-06-22','2026-06-22','6품번·8조건·21숫자. 독립개체·단위 미확정, 대표품번0'],
 ['마이크로크랙 숫자',1,len(image_tables['마이크로크랙_시험기록']),str(image_tables['마이크로크랙_시험기록'].date.min()),str(image_tables['마이크로크랙_시험기록'].date.max()),'52숫자. 측정법·단위·반복 미확정'],
 ['이온교환 원본 셀',1,len(ion),str(ion.date.min()),str(ion.date.max()),'16시험조건×4성분. 전후값완비14조건·대상정화율28값'],
 ['첨가제 농도',1,len(lab),str(lab.date.min().date()),str(lab.date.max().date()),'%rel, 불규칙 측정. 일별 보간 없음'],
 ['전력 월계',14,len(power),power.month.min(),power.month.max(),'월×스테이션'],
 ['월별 보고서',15,len(ppt_monthly),ppt_monthly.month.min(),ppt_monthly.month.max(),'출하검사와 모집단 다름'],
],columns=['자료','파일 수','분석 레코드','시작','종료','레코드 정의']);save(coverage,'data_coverage')
quality=pd.DataFrame([
 ['두께 실측기록의 범위','다른 6품번21값·8조건·1일, 대표0값','단위·반복/개체 미확정으로 공정능력/대표품목 예측은 보류','image_두께_실측기록.csv'],
 ['공정 4조건의 유효 표본','값 존재12,902 / 유효12,864','38행을 활성·양수 조건에서 추가 제외','join_coverage.csv'],
 ['이온교환 이미지 충돌','이미지에만3조건·날짜충돌1조건','연도 추정·날짜 충돌·표시정밀도 별도 보존','image_이온교환_이미지대조.csv'],
 ['검사수량 빈칸',f'{len(daily)-len(valid)}일','기존 prepared는0으로 대체. 결측 유지 및 생산로그 대조','missing_reviewed_days_with_production.csv'],
 ['5월 검사 파일의 4월1일 열','2,979 불량 / 22,951 검사','5월 재집계에서 제외. source Total과 달라짐','inspection_sheet_audit.csv'],
 ['검사 시트 수식 오류',str(sum(x['formula_error_cells'] for x in daily_audit))+'셀','원본 수정 없이 날짜/유형별 숫자 다시 집계','inspection_sheet_audit.csv'],
 ['16번 스테이션 저전압',str(int(sensor[sensor.station.str.startswith('16')].low_voltage.sum()))+'행','<=0.1V를 진단상 제외. 원자료 전압과 대조','station_sensor_diagnostics.csv'],
 ['액 성분 시트 충돌',str(len(conflict_keys))+'키','우선순위 적용, 충돌 원자료 보존','solution_conflicting_sources.csv'],
 ['약품 수동값 음수',str(int(chem_audit.negative_manual_days.sum()))+'일×약품','음수 유지. 순투입과 자동투입을 각각 분석','chemical_reconciliation.csv'],
 ['전류밀도 이미지 산술 불일치',_PREPARED_INPUTS['parameters']['density_example'],'공식에 따라 설정 및 실제평균 밀도 별도 산출','requirements.csv'],
 ['16번 전력계의 저전압 영향','월별 kWh/Ah가 다른 스테이션보다 매우 작음','약품/kWh 비율은 확인용이며 센서 점검 전 효율기준 확정 불가','power_monthly_stations.csv'],
 ['7월30일까지 파일 종료시각',str(chrome.date.max().date()),'자정 통과 이력은 실제 종료일 유지','representative_chrome_events.csv'],
],columns=['점검 항목','규모','처리','근거']);save(quality,'data_quality_issues')
versions={name:__import__(name).__version__ for name in ['pandas','numpy','scipy','sklearn','matplotlib','openpyxl','xlrd']}
(OUT/'environment.json').write_text(json.dumps({'python':sys.version,'platform':platform.platform(),'packages':versions,'seed':SEED,'lags':LAGS,'voltage_floor':VOLTAGE_FLOOR},ensure_ascii=False,indent=2),encoding='utf-8')

def table_html(df,limit=None):
    if limit:df=df.head(limit)
    formatted=df.rename(columns=TABLE_LABELS)
    formatters={'BH 보정 q':lambda x:'<0.001' if x<0.001 else f'{x:.3f}'} if 'BH 보정 q' in formatted else None
    content=formatted.to_html(index=False,escape=True,border=0,na_rep='결측/미확인',float_format=lambda x:f'{x:,.3f}',formatters=formatters)
    content=re.sub(r'([A-Za-z0-9_]+\.csv)',lambda m:f'<a href="tables/{m[1]}">{m[1]}</a>',content)
    return '<div class="table-wrap">'+content+'</div>'
def img_html(name):
    b=base64.b64encode((FIGURES/name).read_bytes()).decode()
    return f'<img class="chart" src="data:image/png;base64,{b}" alt="{html.escape(name)}">'
global_lag=correlations[(correlations.target=='pinhole_pit_pphv')&(correlations['sample']=='common_3_lags')&correlations.feature.isin(heat_features)][['feature','lag_days','n_days','spearman_r','month_weekday_volume_adjusted_r','q_bh']]
best_hold=holdout[holdout.model.ne('Median baseline')].iloc[0];base_hold=holdout[holdout.model.eq('Median baseline')].iloc[0]
performance_note='시험 MAE가 중앙값 기준선보다 낮았습니다. 단일 시험월 결과이며 운영 성능으로 확정하지 않습니다.' if best_hold.mae_pp<base_hold.mae_pp else '시험 MAE가 중앙값 기준선보다 높았습니다. 현 상태에서 예측모형을 운영 기준으로 권고하지 않습니다.'


chemical_gaps = []
chemical_calendar = pd.date_range(chem.date.min(), chem.date.max())
for kind, group in chem.groupby('chemical'):
    for date in chemical_calendar.difference(group.date):
        chemical_gaps.append({'date': date.strftime('%Y-%m-%d'), 'chemical': kind,
                              'status': '일계 행 미기록; 휴무·0투입·누락 미확정'})
save(pd.DataFrame(chemical_gaps), 'chemical_missing_calendar_days')

sections = [
    ('주요 결과', f'''<p>검사수량이 있는 {len(valid)}일을 기준으로 계산했습니다.</p>
<ul><li>검사수량 {reviewed_total:,.0f}개, 불량기록 {total_defects:,.0f}건</li>
<li>핀홀·피트 {pinhole_total:,.0f}건, 전체 불량기록 중 {100*pinhole_total/total_defects:.1f}%</li>
<li>대표품목 크롬 공정기록 {len(representative):,}행</li></ul>
<p>불량기록 비율은 불량기록 건수를 검사수량으로 나눈 값입니다. 한 제품에 여러 불량이 기록될 수 있어 불량 제품의 비율과는 다를 수 있습니다.</p>''' + img_html('01_monthly_defects.png')),
    ('분석대상과 계산 기준', table_html(requirements) + table_html(theme_reference)
     + '<p>테마 문서의 예시 표는 설명용이며 실제 측정값에 포함하지 않습니다.</p>'),
    ('입력자료와 표본 수', table_html(coverage)
     + f'<p>원본 파일 {len(inventory)}개를 사용했습니다.</p>'),
    ('검사 집계', table_html(monthly)
     + '<p>월별 비율은 해당 월의 불량기록 합계를 검사수량 합계로 나누어 계산합니다. 검사수량이 없는 날은 비율 계산에서 제외했습니다.</p>'),
    ('생산·검사 기록의 연결 범위', table_html(joins) + table_html(key_audit)
     + '<p>검사자료에는 품번과 생산묶음 번호(LOT)가 없습니다. 따라서 같은 제품의 생산·검사 기록을 직접 연결할 수는 없고, 날짜별 기록을 비교했습니다.</p>'),
    ('대표품목 공정조건', table_html(item_summary) + img_html('02_representative_conditions.png')
     + table_html(sensor)
     + '<p>같은 제품이 공정을 다시 통과할 수 있어 작업 기록 수와 제품 수는 다릅니다. 전압이 매우 낮게 기록된 경우에는 원래 값과 해당 기록을 제외한 값을 함께 확인했습니다.</p>'),
    ('검사일과 작업일 비교', img_html('03_lag_comparison.png') + table_html(global_lag)
     + '<p>검사 결과를 같은 날, 5일 전, 10일 전의 작업 기록과 비교했습니다. 상관계수(r)는 양수이면 함께 커지는 경향, 음수이면 서로 반대로 변하는 경향을 나타냅니다. 이 결과만으로 불량의 원인을 확정할 수는 없습니다.</p>'),
    ('도금액 성분과 약품 기록', table_html(spec_summary[spec_summary.bath.eq('도금조')])
     + table_html(lab_coverage) + img_html('05_solution_available_lags.png')
     + '<p>도금액 성분은 실제 측정된 날짜만 사용했습니다. 약품량은 원본의 수동 정정값을 포함한 합계와 자동투입량을 각각 계산했습니다.</p>'),
    ('월별 약품량과 전력량', table_html(theme2_monthly)
     + '<p>월별 약품량을 전기에너지(kWh)와 전류가 흐른 양(Ah)으로 각각 나누어 비교했습니다. Nr.6는 원본에 적힌 첨가제 구분입니다. 이 비율만으로 적정 투입량이나 절감 효과를 확정할 수는 없습니다.</p>'),
    ('도금액 성분과 작업조건 비교', table_html(feedback, 18)
     + '<p>품목·설비별 하루 평균 작업조건과 도금액 성분을 날짜별로 비교했습니다. 각 설비가 어떤 도금액 탱크를 사용했는지 확인되지 않아 특정 성분의 영향을 확정할 수는 없습니다.</p>'),
    ('예측 결과 확인', table_html(cv_summary) + table_html(holdout)
     + '<p>과거 자료로 두 예측 방법(Ridge, Random Forest)을 학습하고 2026년 5월의 실제 결과와 비교했습니다. 평균 예측 오차(MAE)는 작을수록 실제 값에 가깝습니다. 실제 5%를 7%로 예측했다면 차이는 2%p입니다. 전체 품목의 날짜별 기록을 평가한 결과로, 개별 제품의 불량 판정에는 사용할 수 없습니다.</p>'
     + img_html('04_holdout_predictions.png')),
    ('월별 보고서와 검사 집계의 차이', table_html(monthly_comparison[
        ['month','defects','reviewed','total_pphv','defects_ppt','production_ppt','ppm_reported']])
     + '<p>월별 보고서와 출하검사는 집계 대상이 달라 나란히 비교했습니다. 두 자료의 수치를 합치지는 않았습니다.</p>'),
    ('계산 결과 점검', table_html(quality) + table_html(verification)),
    ('결과 파일', '<p>세부 계산표는 tables, 그래프는 figures 폴더에 있습니다. 입력파일 해시는 tables/source_files_sha256.csv, 실행 환경은 environment.json에 저장합니다.</p>'),
]
if additional_result['input_rows']:
    sections.insert(0, ('새로 추가한 자료의 분석', additional_result['summary_html']))

css = '''body{font-family:"Malgun Gothic",Arial,sans-serif;color:#222;background:#f5f6f7;margin:0;line-height:1.7}
main{max-width:1120px;margin:28px auto;padding:36px;background:white}h1{font-size:30px}h2{font-size:22px;margin-top:40px;border-bottom:1px solid #ddd;padding-bottom:8px}
p,li{font-size:15px}.table-wrap{overflow-x:auto;margin:18px 0}table{border-collapse:collapse;width:100%;font-size:12px}th,td{padding:9px;border-bottom:1px solid #ddd;text-align:left;vertical-align:top}
th{background:#355c7d;color:white}.chart{max-width:100%;height:auto}a{color:#355c7d}nav{display:flex;gap:8px 18px;flex-wrap:wrap}'''
nav = '<nav>' + ''.join(f'<a href="#s{i}">{html.escape(title)}</a>' for i,(title,_) in enumerate(sections)) + '</nav>'
report = ('<!doctype html><html lang="ko"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1">'
          '<title>한국로텍 데이터 분석 결과</title><style>' + css + '</style></head><body><main><h1>한국로텍 데이터 분석 결과</h1>'
          + f'<p>출하검사 기간: {valid.date.min():%Y-%m-%d}~{valid.date.max():%Y-%m-%d}</p>' + nav
          + ''.join(f'<section id="s{i}"><h2>{html.escape(title)}</h2>{content}</section>' for i,(title,content) in enumerate(sections))
          + '</main></body></html>')
(OUT / '한국로텍_분석보고서.html').write_text(report, encoding='utf-8')
shutil.copy2(_nb_file, OUT / '한국로텍_데이터분석.ipynb')
_output_manifest = TABLES / 'output_manifest.csv'
pd.DataFrame([{'file':p.relative_to(OUT).as_posix(),'bytes':p.stat().st_size}
              for p in sorted(OUT.rglob('*')) if p.is_file() and p != _output_manifest
              and not p.name.startswith('~$')]).to_csv(_output_manifest, index=False, encoding='utf-8-sig')
display(Markdown('**분석 완료** · [분석 보고서 열기](분석결과/한국로텍_분석보고서.html) · 세부 계산표와 그래프는 왼쪽 파일 목록의 `분석결과` 폴더에 있습니다.'))
